# **CardiAg: Cardiac Phase Analysis**

Pipeline for the analysis of behavioral events from the CardiAg Intentional Binding task with respect to cardiac phase derived from the ECG signal, including circular and binary analysis of cardiac phase. 

Authors: Marta Gerosa

Created on: 24 January 2025

Last update (by M. Gerosa): 21 April 2026

Before using the pipeline, we recommend downloading the already preprocessed ECG files and store them in the `data/derivatives/ecg-preproc` folder. Otherwise, if you wish to preprocess your own raw ECG data, you can run the `CardiAg_ecg_preproc.ipynb` on the subject-level BIDS-compatible `*_physio.tsv.gz` files. 

The following packages are required: `rpy2` (>= 3.5). Importantly, a working installation of R is also required to run the circular analysis section. 

## **Pipeline structure**

The following steps are included:

1. **Behavioral data import and merge into long format**: load the preprocessed behavioral data (`_beh-preproc.tsv`) from the `/derivatives/beh-preproc/sub-xx/` folder for each subject and merge all of them into one DataFrame `behpreproc_long` in long format. Please note that the behavioral data file should have one row per each trial, containing task-relevant information (e.g., condition, n trial etc.) and - at least - one column containing timestamps (in sec) for a task event of interest (e.g., keypress or visual stimulus onset). 
2. **Preprocessed ECG data import and conversion**: load the preprocessed ECG data (`_ecg-preproc.json`) from the `/derivatives/ecg-preproc/sub-xx/` folder and its metadata (`_physio.json`) from the BIDS raw data folder for each subject. Extract the sample idx for R-peaks and other relevant QRS features, then convert them into timestamps (in sec). 
3. **ECG features extraction around task event**: extract the ECG features around a (user-defined) task event, mainly R-peaks, T-wave offsets and other QRS features needed for systole/diastole segmentation, on a trial-by-trial basis. These features are also used to calculate event-related R-R interval duration (in sec), instantaneous HR (in bpm and Hz), temporal and angular position within the cardiac cycle (for circular analysis), systole (PEP+EP) and diastole duration, as well as for binning events into systole, diastole or non-defined intervals (for binary analysis). If `equalize_bins` is `True`, additional segmentation of systole/diastole and binning can be performed using the equal window length approach. This procedure is performed for both the action trials (`beh_ecg_action`) and tone trials (`beh_ecg_tone`).
4. **(Optional) T-offset recomputation using Trapezium Area (TRA) approach**: if `recompute_toff_tra` is set to `True`, (optionally) recompute T-wave offsets using the Trapezium Area approach (TRA; Vázquez-Seisdedos et al., 2011) and adjust systole and diastole segmentation accordingly. This section is useful to compare T-offset delineation performed by NeuroKit2 `ecg_delineate(method='dwt')` during ECG preprocessing with the one performed using the TRA approach, especially in noisy signal, and later decide which of the systole/diastole segmentations to use. For selected participants, the T-offset and systole/diastole segmentation is recalculated using the TRA approach, and the final dataset comprising behavioral data and corresponding ECG features and individual systole/diastole segmentation are stored in the `task-CardiAgIBTask_beh_ecg_action.tsv` and `task-CardiAgIBTask_beh_ecg_tone.tsv` for action and tone trials, respectively. Furthermore, descriptives for the individual cardiac phase durations are provided. This corresponds to **Supplementary Results sections 1.2.1 and 1.2.2**, with the respective **Figures S1 & S2**. 
5. **(Optional) individual plot of ECG around task event with sys/dia segmentation**: if `plot_ecg2event` is set to `True`, create an interactive plot showing the individual ECG trace for a user-defined participant with the onset of the task event on each trial, block (optional) and condition, with shadowed areas corresponding to systole (orange) and diastole (blue) intervals. It can be further specified whether to display the systole vs. diastole segmentation based on NeuroKit2 'DWT' approach or the re-computed TRA approach. 
6. **Binary analysis - systolic vs. diastolic ratio of task event [Results 2.2.2]**: perform binary analysis of individual cardiac phases (systole vs. diastole) with respect to a task event, by computing the phase-specific proportion of task events, relative to all trials, normalized by the proportion of the respective cardiac phase in the entire RR interval. In group-level analysis, normalized systolic and diastolic ratios are tested using a two-sided paired t-test. If a cardiac effect is present, the phase-specific ratio value is expected to be > 1, indicating over-proportional accumulation of task events in the respective phase. This corresponds to **Results section 2.2.2** with corresponding **Figure 3b**. 
7. **Binary analysis - cardiac phase binning of intentional binding [Results 2.2.3 & 2.5.3]**: compute cardiac phase differences in action binding or tone binding (i.e., dependent variable), respectively, by binning action or tone into systole/diastole; this corresponds to **Results section 2.2.3** with corresponding **Figures 3c, 3d & 3e**. In addition, perform correlation analysis between the individual normalized systolic and diastolic ratios for action/tone trials and the corresponding intentional binding measure (i.e., action binding and tone binding, respectively); this corresponds to **Results section 2.5.3** with corresponding **Figures 7a & 7b**. The participant-level means of action/tone binding binned by cardiac phase are saved under `task-CardiAgIBTask_ecg_actbinding_avg.tsv` and `task-CardiAgIBTask_ecg_tonebinding_avg.tsv`. 
8. **Circular analysis of task events across cardiac cycle [Results 2.2.1]**: first, perform 1st-level circular analysis (within-subject) of task event (i.e., action onset) across cardiac cycle, testing the individual circular means against null distribution using participant-level Rayleigh test. Second, perform 2nd-level circular analysis (between-subject) of task event (i.e., action onset) across cardiac cycle, testing the group-level mean resultant length ϱ against the null hypothesis of uniform distribution using a Rayleigh test. To calculate 95% confidence intervals and significance, a non-parametric bootstrap analysis (10000 iterations, with replacement) is performed. The resulting group-level circular plot with 2nd-level circular mean and mean resultant length, individual circular means, average circular distribution with 95% CI and significant segments, as determined by bootstrap analysis, is saved. This corresponds to **Results section 2.2.1** with **Figure 3a**. Lastly, condition-wise pairwise comparisons of circular distributions of action onsets according to intentional binding condition are computed, as corresponding to **Extended Data Figure 1**.
9. **Pre- and post-event R-R interval changes [Results 2.5.1]**: analyze changes in the duration of R-R intervals before and after the action onset, indicative of cardiac deceleration and/or acceleration. In addition, perform slope analysis of cardiac deceleration (z-scored) preceding action onset (i.e., from T-1 to T0), and of cardiac acceleration (z-scored) following action onset (i.e., from T0 to T+1), between intentional binding conditions. The resulting dataframes with R-R interval durations preceding and following action onset are saved under `task-CardiAgIBTask_preact_cardiac_deceleration.tsv` and `task-CardiAgIBTask_postact_cardiac_acceleration.tsv`. This corresponds to **Results section 2.5.1**, with **Figure 6a, 6b & 6c**. 

In [1]:
############## Import modules ##############

import pandas as pd
import numpy as np
import os
import json
import math
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from IPython.display import Markdown as md
from tqdm import tqdm
import pickle
import pingouin as pg
from statsmodels.stats.multitest import fdrcorrection
import matplotlib.patches as patches

# Import custom module (`cardiacphase_functions.py.py` should be in same directory)
import cardiacphase_functions as cardphase_utils

In [2]:
############## Settings: optional pipeline steps ##############

# For all sections: choose the task event type that will be analysed with respect to the cardiac cycle,
# by specifying the column name with timestamps (in sec) and the abbreviation to be appended in the variable names

# Action onset analysis
event_col = 'keypress_onset'    # define column of beh dataset with timestamps (in sec) of task event onset
abbrev = 'act'                  # define abbreviation of task event for further column names in cardiac phase analysis

# Tone onset analysis
event_col_tone = 'tone_onset'    # define column of beh dataset with timestamps (in sec) of task event onset
abbrev_tone = 'tone'             # define abbreviation of task event for further column names in cardiac phase analysis

# For all sections: specify your column names for participant ID, condition, trial and block (the latter is optional)
column_map = {
    'participant': 'subjID',
    'condition': 'condition',
    'trial': 'n_trial',
    'block': 'n_block' # remove this key-value pair if block is not available
}

# For S4: set whether (optional) recomputation of T-offset using Trapezium Area (TRA) approach is needed
# to correct/compare T-offsets, especially with noisy signal where NeuroKit2 nk.ecg_delineate(method='dwt') may have failed
recompute_toff_tra = True

# For S5: set whether (optional) individual plot of ECG around task event per block/trial is needed
plot_ecg2event = True

## **Settings: R-Python interface**

Please note that, in order to use this pipeline (in particular, to conduct circular analysis of cardiac phase), a working installation of R is needed (>= 4.0). In the section below, the user should specify the path to their R installation and their rpy2 installation - the former will likely be under the `Program Files/R` folder, the latter under the dedicated virtual env folder (check under `Lib/site-packages/`). 

* When using the pipeline for the first time, set `rinstall_firsttime` to `True`. Otherwise, keep it as `False`. 

Further information on the rpy2 module and how to interface with R in Python can be found here: https://rpy2.github.io/

In [ ]:
############## Import modules to interact with R packages in Python ##############

# Path to user-defined R installation
os.environ['R_HOME'] = 'C:/Program Files/R/R-4.4.0' # define path to R
os.environ['R_USER'] = 'C:/Users/.../AppData/Local/miniforge3/envs/cardiag_env/Lib/site-packages/rpy2' # define path to rpy2 in your virtual environment

# Set to TRUE the first time that the script is run to download the necessary R packages, otherwise FALSE
rinstall_firsttime = False


# Import or install R packages that will be used 
from rpy2.robjects.packages import importr
import rpy2.robjects.packages as rpackages
from rpy2.robjects.vectors import StrVector
from rpy2.robjects import pandas2ri
import rpy2.robjects as ro
from rpy2 import robjects

# Activate pandas-R conversion
pandas2ri.activate()

# Import useful R packages
utils = rpackages.importr('utils') # import R 'utils' package
grdevices = importr('grDevices')
graphics = importr('graphics')
lme4 = importr('lme4')
base = importr('base')

if rinstall_firsttime:
    utils.chooseCRANmirror(ind=1) # Select the first mirror for R packages
    packnames = ('circular') # R packages to install/import
    # Selectively install what needs to be installed
    names_to_install = [x for x in packnames if not rpackages.isinstalled(x)]
    if len(names_to_install) > 0:
        utils.install_packages(StrVector(names_to_install))

# Import "circular" package
circular = rpackages.importr('circular')

## **1. Behavioral data import and merge into long format**

This section imports the preprocessed behavioral data `_beh-preproc.tsv` for each participant from the `/derivatives/beh-preproc` folder, turning them into a DataFrame (`subj_behpreproc`). Then, all the participant-specific DataFrames are concatenated together into the `behpreproc_long` DataFrame in long format, i.e. one row per trial, all participants into one file. In detail: 
- Iterate through participant IDs specified in a list.
- Define the name and directory path for the `_beh-preproc.tsv` file containing preprocessed behavioral data for each participant. 
- Import the TSV file and store as a DataFrame (`subj_behpreproc`).
- Merge the participant-specific `subj_behpreproc` DataFrames into one unique `behpreproc_long` DataFrame in long format. 
- Define the base directory `derivatives/cardiac-phase-analysis` folder for storage of results.

In [ ]:
################## 1. Behavioral data import and merge into long format ##################

# Sub-16 excluded due to extreme JE distribution, sub-14 excluded due to missing HW triggers
# Sub-33 excluded due to outlier IB measures 
participant_ids = [1, 2, 3, 5, 6, 12, 13, 15, 17, 18, 19, 
                   20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 
                   31, 32, 34, 35, 36, 37, 38, 39, 41, 42,          
                   43, 44, 45, 46, 47, 48, 49, 51, 53, 54, 55, 57]  

# Specify the data path info
wd = r'.\data'                      # directory of data storage
exp_name = 'CardiAgIBTask'                 
datatype_name = 'beh'               # current datatype folder according to BIDS

# Initialize empty list to store dfs with beh preprocessed data for all participants
subj_dfs = []

# Iterate through each participant
for subj in participant_ids:

    subj_id = 'sub-' + str(subj) # participant ID (in BIDS format)
    behpreproc_fname = f'{subj_id}_task-{exp_name}_beh-preproc.tsv'

    # Merge information into complete datapath
    behpreproc_dir = os.path.join(wd, 'derivatives', 'beh-preproc', subj_id, 
                                  datatype_name, behpreproc_fname)

    # Read TSV file into a dataframe
    subj_behpreproc = pd.read_csv(behpreproc_dir, sep='\t', header=0)

    # Reset the index to avoid having duplicate index columns and append to list
    subj_behpreproc.reset_index(drop=True, inplace=True)
    subj_dfs.append(subj_behpreproc)

# Create a long data frame with beh preprocessed data from all participants
behpreproc_long = pd.concat(subj_dfs, ignore_index=True)
behpreproc_long.rename(columns={'Unnamed: 0': 'subj_index'}, inplace=True) # comment if unnecessary

# Subselect only relevant behavioral data for cardiac-beh analysis
behpreproc_long = behpreproc_long[['subjID', 'condition', 'n_block', 'n_trial', 
                                   'real_report_diff_act_deg', 'real_report_diff_tone_deg', 'JE_act', 'JE_tone', 
                                   'trial_onset', 'keypress_onset', 'tone_onset']].copy()


####### Create general directory for saving output from cardiac phase analysis #######

# Check if the BIDS directory 'derivatives/cardiac-phase-analysis/' exists, if not create it
cardphase_folder = 'cardiac-phase-analysis'
cardphase_dir = os.path.join(wd, 'derivatives', cardphase_folder)
if not os.path.exists(cardphase_dir):
    os.makedirs(cardphase_dir)

## **2. Preprocessed ECG data import and conversion**

This section import the preprocessed ECG data (`_ecg-preproc.json`) from the `/derivatives/ecg-preproc/sub-xx/` folder, as well as the corresponding physio metadata (`_physio.json`) from the raw data folder, in order to extract the sample idx for R-peaks and other relevant QRS features and covert them into timestamps (in sec). The following ECG features are derived from the ECG preprocessing file: R-peaks, Q-wave onsets (alternatively, Q-peaks), S-wave offsets, and T-wave offsets. 

In [5]:
############## 2. Preprocessed ECG data import and conversion ##############

# Initialize empty dict to store corrected rpeaks idx for all participants
subj_qrst = {}

# Iterate through each participant
for subj in participant_ids:

    ############## Load ECG signal metadata ##############

    # Specify the name and directory of physio JSON file (for metadata)
    json_md_fname = f'{subj_id}_task-{exp_name}_physio.json'
    json_md_dir = os.path.join(wd, subj_id, datatype_name, json_md_fname)

    # Extract physio metadata from JSON file
    fjson = open(json_md_dir)
    ecg_metadata = json.load(fjson) # load metadata from _physio.json
    sfreq = ecg_metadata['SamplingFrequency'] # get sampling rate frequency


    ############## Load preprocessed ECG data (incl. R-peaks, QRS idx) ##############

    subj_id = 'sub-' + str(subj)  # Participant ID (in BIDS format)
    json_ecg_fname = f'{subj_id}_task-{exp_name}_ecg-preproc.json'
    json_ecg_dir = os.path.join(wd, 'derivatives', 'ecg-preproc', 
                                subj_id, datatype_name , json_ecg_fname)

    # Check if JSON file exists and, if so, load it
    if os.path.exists(json_ecg_dir):
        with open(json_ecg_dir, 'r') as json_ecg_preproc:
            ecg_preproc_data = json.load(json_ecg_preproc)
        
        ####### Extract R-peaks idx #######

        # Extract rpeaks values: in order, if present, prefer the manually corrected 
        # over automatedly corrected, and these over the uncorrected ones
        preferred_keys = ['ECG_R_Peaks_ManualCorr', 'ECG_R_Peaks_AutoCorr', 'ECG_R_Peaks_Uncorr']

        # Get the R-peaks indices based on the preference list
        for key in preferred_keys:
            rpeaks_idx = ecg_preproc_data.get('rpeaks', {}).get(key)
            if rpeaks_idx:
                # If one of the indices is found, break the loop
                break

        # Append the R-peaks values to the subj dict
        subj_qrst[subj_id] = {'rpeaks_idx': rpeaks_idx}

        ####### Extract QRS features idx #######

        # Extract QRS features: T-offsets, S-offsets, Q-onsets and Q-peaks
        # for systole-diastole segmentation and delineation of pre-ejection period (PEP)
        t_offsets_idx = ecg_preproc_data['qrs']['ECG_T_Offsets']
        s_offsets_idx = ecg_preproc_data['qrs']['ECG_R_Offsets']  # "ECG_R_Offsets" in NK indicate S-wave offset in QRS complex
        q_onsets_idx = ecg_preproc_data['qrs']['ECG_R_Onsets']    # "ECG_R_Onsets" in NK indicate Q-wave onsets in QRS complex
        q_peaks_idx = ecg_preproc_data['qrs']['ECG_Q_Peaks']

        # Append the QRS features values in sample indices to the subj dict
        subj_qrst[subj_id] = {'rpeaks_idx': rpeaks_idx,         # sample idx of R-peaks
                              't_offsets_idx': t_offsets_idx,   # sample idx of T-wave offsets (end of sys)
                              's_offsets_idx': s_offsets_idx,   # sample idx of S-wave offsets (end of PEP) -> ECG_R_Offsets in NK
                              'q_onsets_idx': q_onsets_idx,     # sample idx of Q-wave onsets (start of PEP & end of dia) -> ECG_R_Onsets in NK
                              'q_peaks_idx': q_peaks_idx}       # sample idx of Q-peaks (alternative end of dia for some subj)
        
        ####### Transform idx of R-peaks and QRS features to timestamps #######

        # Transform idx into timestamps (in sec)
        rpeaks_time = [idx / sfreq for idx in rpeaks_idx]
        t_offsets_time = [idx / sfreq for idx in t_offsets_idx if idx is not None]
        s_offsets_time = [idx / sfreq for idx in s_offsets_idx if idx is not None]
        q_onsets_time = [idx / sfreq for idx in q_onsets_idx if idx is not None]
        q_peaks_time = [idx / sfreq for idx in q_peaks_idx if idx is not None]

        subj_qrst[subj_id]['rpeaks_s'] = rpeaks_time        # timestamps of R-peaks
        subj_qrst[subj_id]['t_offsets_s'] = t_offsets_time  # timestamps of T-wave offsets
        subj_qrst[subj_id]['s_offsets_s'] = s_offsets_time  # timestamps of S-wave offsets (end of PEP) -> ECG_R_Offsets in NK
        subj_qrst[subj_id]['q_onsets_s'] = q_onsets_time    # timestamps of Q-wave onsets (start of PEP & end of dia) -> ECG_R_Onsets in NK
        subj_qrst[subj_id]['q_peaks_s'] = q_peaks_time      # timestamps of Q-peaks (alternative end of dia for some subj)

    else:
        print(f"ECG preprocessing JSON file not found for {subj_id}")

## **3. ECG features extraction around task event**

This section extracts the ECG features around a (user-defined) task event, mainly R-peaks, T-wave offsets and other QRS features needed for segmentation into systole and diastole, on a trial-by-trial basis. Then, proceeds to calculate the duration of the R-R interval around the task events, the instantaneous HR, the temporal and angular position of the task event with respect to one cardiac cycle (for circular analysis), the duration of systole (PEP+EP) and diastole, and lastly determines whether the event falls into systole, diastole or non-defined interval (for binary analysis). If `equalize_bins` is `True`, additional segmentation and binning can be performed using the equal window length approach (see Esra Al's work). In detail: 

- The user must specify the column name contaning timestamps - in sec - for the task event, using the function argument `event_col` (e.g., 'keypress_onset'), as well as an abbreviated name for the event type (e.g., 'act')
- For each trial, the relevant ECG features surrounding a user-defined task event of interest are extracted: R-peaks preceding (R-peak pre) and following (R-peaks post) the task event, Q-wave onsets and S-wave offsets around R-peak pre to delineate the onset and offset of the systolic PEP, T-wave offsets to determine the systole end and duration, and Q-wave onsets around R-peak post to determine the diastole end. Further, the diastole onset is determined at +50ms after T-wave offset. 
- These ECG features are used to determine on a trial-by-trial basis: R-R interval (in sec), instantaneous HR (in bpm and Hz), time difference of the task event with respect to the R-peak pre, angular difference (in rad) of the task event with respect to the R-peak pre (for circular analysis). 
- Lastly, the cardiac phases (systole, diastole) are segmented, deriving their duration and determining whether the task event falls in each respective phase or in a non-defined one (i.e., PEP, 50ms buffer, post-diastole), by assigning a binary 0/1 value. 
- Optionally, if the argument `equalize_bins` is `True`, an additional segmentation of systole and diastole is computed using windows of equal length (i.e., sys duration = dia duration), starting from R-peak pre or ending with R-peak post respectively, which is then used to determine the binary 0/1 value of whether the task event falls in each respective phase.  

In the context of the CardiAg intentional binding task, this procedure is performed for both the action trials (`beh_ecg_action`) and tone trials (`beh_ecg_tone`).

In [6]:
############## 3. ECG features extraction around task event ##############

# Define a custom function for ECG features extraction around task event of interest:
# R-peaks, T-offsets, Q-onsets and S-offsets used to segment into systole/diastole
# and prepare calculations for cardiac circular & binary analysis

def extract_ecg_features_event(event_col, abbrev, behpreproc_long=behpreproc_long, subj_qrst=subj_qrst, 
                               participant_col=column_map['participant'], equalize_bins=False):
    """
    Extract ECG features around a specified event of interest and create a preprocessed df 
    per trial and participants, used in later cardiac phase analysis. 

    Parameters
    -----------
    event_col : str
        Column name from `behpreproc_long` containing the timestamps in sec for the event of interest
        (e.g., 'keypress_onset'), on which cardiac phase analysis will be conducted. 
    abbrev : str
        Abbreviated name for the event type (e.g., 'act'), used as suffix for all the derived columns.
    behpreproc_long : DataFrame
        DataFrame containing behavioral data of all participants in long format (one row per trial) 
        and events timestamps, created in earlier steps. Default is `behpreproc_long`. 
    subj_qrst : dict
        Dictionary of ECG features in idx and sec per participant, created in earlier steps. Default is `subj_qrst`. 
    participant_col : str
        Column name storing the participant IDs.   
    equalize_bins : bool, default False
        If True, further systolic and diastolic windows of equal length are computed, starting from R-peak pre 
        and ending with R-peak post, respectively. 

    Returns
    --------
    DataFrame:
        Concatenated DataFrame containing the original behavioral data for all participants plus columns 
        with ECG features relative to the user-defined task event, including:
            - `Rpeak_pre_{abbrev}`: timestamp (in sec) of R-peak pre (i.e., just before task event)
            - `Rpeak_post_{abbrev}`: timestamp (in sec)  of R-peak post (i.e., just after task event)
            - `Toffset_{abbrev}`: timestamp (in sec) of T-wave offset within R-peak pre and post range (i.e., end of sys)
            - `Qonset_pre_{abbrev}`: timestamp (in sec) of Q-wave onset just before R-peak pre (i.e., start of sys PEP)
            - `Soffset_{abbrev}`: timestamp (in sec) of S-wave offset within R-peak pre and post range (i.e., end of sys PEP)
            - `Qonset_post_{abbrev}`: timestamp (in sec) of Q-wave onset just before R-peak post (i.e., end of dia)
            - `RR_s_{abbrev}`: RR interval around event onset (in sec)
            - `HR_1perMin_{abbrev}`:  instantaneous HR per min (in bpm) based on RR interval length 
            - `HR_1perSec_{abbrev}`: instantaneous HR per sec (in Hz) based on RR interval length 
            - `diff_event_Rpeak_{abbrev}`: time difference (in sec) from task event to the preceding R-peak 
            - `event_ecg_radian_{abbrev}`: angular position of task event (in radians) relative to cardiac cycle (RR interval)
            - `sys_dur_total_{abbrev}`: total systole duration (in sec), calculated from Q-onset pre to T-offset (PEP + EP, i.e. QT interval)
            - `sys_dur_pep_{abbrev}`: duration of systolic PEP (in sec), calculated from Q-onset pre to S-offset 
            - `sys_dur_ep_{abbrev}`: duration of systolic EP (in sec), calculated from S-offset to T-offset
            - `dia_onset_{abbrev}`: timestamp (in sec) of diastole onset, calculated as +50 ms after T-offset
            - `dia_dur_{abbrev}`: total diastole duration (in sec), calculated from +50ms after T-offset to Q-onset post
            - `{abbrev}_sys`: binary value 0/1 of whether the task event falls into systolic window
            - `{abbrev}_dia`: binary value 0/1 of whether the task event falls into diastolic window
            - `{abbrev}_pep`: binary value 0/1 of whether the task event falls into systolic PEP window (non-defined)
            - `{abbrev}_buffer`: binary value 0/1 of whether the task event falls into sys-dia 50ms buffer window (non-defined)
            - `{abbrev}_postdia`: binary value 0/1 of whether the task event falls into post-diastole window until next R-peak (non-defined)
        Optional columns (if equalize_bins is True):
            - `sys_dur_equal_{abbrev}`: duration of systolic window (in sec), calculated from R-peak pre to T-offset, of equal length as the diastolic window
            - `dia_onset_equal_{abbrev}`: timestamp (in sec) of diastole onset, calculated as equal length to systole starting from R-peak post
            - `nondef_dur_equal_{abbrev}`: duration of non-defined cardiac window between systole and diastole (in sec)
            - `{abbrev}_sys_equal`: binary value 0/1 of whether the task event falls into systolic window of equal length
            - `{abbrev}_dia_equal`: binary value 0/1 of whether the task event falls into diastolic window of equal length

    """
    
    beh_ecg_dfs = [] # initialize empty list to store dfs for all participants

    # Extract relevant ECG features for the current participant
    for subj in behpreproc_long[participant_col].unique():
        subj_ecg_data = subj_qrst.get(f'sub-{subj}', {})
        rpeaks_s = subj_ecg_data.get('rpeaks_s', [])
        t_offsets_s = subj_ecg_data.get('t_offsets_s', [])
        s_offsets_s = subj_ecg_data.get('s_offsets_s', [])
        q_onsets_s = subj_ecg_data.get('q_onsets_s', [])
        q_peaks_s = subj_ecg_data.get('q_peaks_s', [])

        # Filter df with preprocessed beh data for current participant
        subj_beh_ecg = behpreproc_long[behpreproc_long[participant_col] == subj].copy()

        # Specify relevant column names for ecg-beh analysis based on the event abbreviation
        columns = {
            'Rpeak_pre': f'Rpeak_pre_{abbrev}', 'Rpeak_post': f'Rpeak_post_{abbrev}',
            'Toffset': f'Toffset_{abbrev}', 'Qonset_pre': f'Qonset_pre_{abbrev}',
            'Soffset': f'Soffset_{abbrev}', 'Qonset_post': f'Qonset_post_{abbrev}',
            'RR_s': f'RR_s_{abbrev}', 'HR_1perMin': f'HR_1perMin_{abbrev}', 
            'HR_1perSec': f'HR_1perSec_{abbrev}', 'diff_event_Rpeak': f'diff_event_Rpeak_{abbrev}',
            'event_ecg_radian': f'event_ecg_radian_{abbrev}', 'sys_dur_total': f'sys_dur_total_{abbrev}', 
            'sys_dur_pep': f'sys_dur_pep_{abbrev}', 'sys_dur_ep': f'sys_dur_ep_{abbrev}',
            'dia_onset': f'dia_onset_{abbrev}', 'dia_dur': f'dia_dur_{abbrev}', 
            f'{abbrev}_sys': f'{abbrev}_sys', f'{abbrev}_dia': f'{abbrev}_dia',
            f'{abbrev}_pep': f'{abbrev}_pep', f'{abbrev}_buffer': f'{abbrev}_buffer', 
            f'{abbrev}_postdia': f'{abbrev}_postdia'}
      
        # Add extra columns if equalize_bins is True
        if equalize_bins:
            columns.update({
                'sys_dur_equal': f'sys_dur_equal_{abbrev}', 'dia_onset_equal': f'dia_onset_equal_{abbrev}',
                'nondef_dur_equal': f'nondef_dur_equal_{abbrev}', f'{abbrev}_sys_equal': f'{abbrev}_sys_equal', 
                f'{abbrev}_dia_equal': f'{abbrev}_dia_equal'})

        subj_beh_ecg = subj_beh_ecg.assign(**{col: np.nan for col in columns.values()})

        # Iterate through each row to extract/calculate various ECG features relative to the task event
        for index, row in subj_beh_ecg.iterrows():
            event_time = row[event_col] # specify column with timestamps for the task event of interest
            
            # Find R-peak pre (just before event) and post (just after event)
            rpeak_pre = max([peak for peak in rpeaks_s if peak <= event_time], default=np.nan)
            rpeak_post = min([peak for peak in rpeaks_s if peak > event_time], default=np.nan)
            subj_beh_ecg.at[index, columns['Rpeak_pre']] = rpeak_pre        # R-peak just before task event
            subj_beh_ecg.at[index, columns['Rpeak_post']] = rpeak_post      # R-peak just after task event

            if not np.isnan(rpeak_pre) and not np.isnan(rpeak_post): 

                # Find timestamp of T-wave offset within R-peak pre and post range (i.e., end of sys)
                t_offset = next((toff for toff in t_offsets_s if rpeak_pre < toff < rpeak_post), np.nan)
                subj_beh_ecg.at[index, columns['Toffset']] = t_offset if t_offset else np.nan
                
                # Find timestamp of Q-wave onset (R-onset in NeuroKit2) just before R-peak pre (i.e., start of sys PEP)
                # for sub-12 & sub-48, use Q-peaks instead of Q-onsets for diastole end (mis-detection)
                if subj in [12, 48]: 
                    q_onset_pre = max([qon for qon in q_peaks_s if qon < rpeak_pre], default=np.nan) 
                else: 
                    q_onset_pre = max([qon for qon in q_onsets_s if qon < rpeak_pre], default=np.nan) 
                if not np.isnan(q_onset_pre) and (rpeak_pre - q_onset_pre) > 0.15:  
                    q_onset_pre = np.nan  # clean up for incorrectly assigned Q-onsets, spanning over physiologically plausible window of 150ms before R-peak
                # for sub-5, use the average Q-onset to R-peak pre duration to interpolate trials where NeuroKit2's QRS delineation failed
                if subj in [5] and np.isnan(q_onset_pre): 
                    mean_q2r_pre_sub5 = 0.04578881987576095   # average Q-onset pre to R-peak pre duration for valid trials of sub-5
                    q_onset_pre = round((rpeak_pre - mean_postdia_sub5),3)
                subj_beh_ecg.at[index, columns['Qonset_pre']] = q_onset_pre if q_onset_pre else np.nan

                # Find timestamp of S-wave offset (R-offset in NeuroKit2) within R-peak pre and post range (i.e., end of sys PEP)
                s_offset = next((soff for soff in s_offsets_s if rpeak_pre < soff < rpeak_post), np.nan)
                subj_beh_ecg.at[index, columns['Soffset']] = s_offset if s_offset else np.nan

                # Find timestamp of Q-wave onset (R-onset in NeuroKit2) just before R-peak post (i.e., end of dia)
                # for sub-12 & sub-48, use Q-peaks instead of Q-onsets for diastole end (mis-detection)
                # for sub-5, use the average Q-onset to R-peak post duration to interpolate trials where NeuroKit2's QRS delineation failed
                if subj in [12, 48]: 
                    q_onset_post = max([qons for qons in q_peaks_s if rpeak_pre < qons < rpeak_post], default=np.nan) 
                elif subj in [5]:
                    mean_postdia_sub5 = 0.045527607361965934    # average post-diastole duration (i.e., from Q-onset post to R-peak post) for valid trials of sub-5
                    q_onset_post = max([qons for qons in q_onsets_s if rpeak_pre < qons < rpeak_post], default=np.nan) 
                    if np.isnan(q_onset_post): 
                        q_onset_post = round((rpeak_post - mean_postdia_sub5),3)
                else: 
                    q_onset_post = max([qons for qons in q_onsets_s if rpeak_pre < qons < rpeak_post], default=np.nan) 
                subj_beh_ecg.at[index, columns['Qonset_post']] = q_onset_post if q_onset_post else np.nan
                
                # Calculate RR interval around event onset (in sec)
                rr_interval = rpeak_post - rpeak_pre    # RR interval as distance between R-peaks pre- and post-event
                subj_beh_ecg.at[index, columns['RR_s']] = round(rr_interval,3)

                # Calculate instantaneous HR per min (in bpm) and per sec (in Hz) based on RR interval length 
                hr_bpm = 60 / rr_interval   # instantaneous HR in bpm
                hr_hz = 1 / rr_interval     # instantaneous HR in Hz 
                subj_beh_ecg.at[index, columns['HR_1perMin']] = hr_bpm 
                subj_beh_ecg.at[index, columns['HR_1perSec']] = hr_hz

                # Calculate time difference (in sec) from task event to the preceding R-peak (Rpeak_pre)
                diff_rpeak_s = event_time - rpeak_pre
                subj_beh_ecg.at[index, columns['diff_event_Rpeak']] = round(diff_rpeak_s,3)

                # Calculate angular position of task event (in rad) relative to cardiac cycle 
                # Cardiac cycle stretches from R-peak pre to R-peak post relative to instantaneous HR 
                ecg_radian = 2 * np.pi * hr_hz * diff_rpeak_s
                subj_beh_ecg.at[index, columns['event_ecg_radian']] = ecg_radian


                ############## ECG features for binary analysis ##############

                ##### Systole #####
                
                # Define total systole interval as comprising pre-ejection phase (PEP) + ejection phase (EP) -> QT interval
                # PEP (not included in mechanical systole) from Q-onset to S-wave offset, EP (left ventricular mechanical systole) from S-wave offset to T-wave offset
                sys_dur_tot = t_offset - q_onset_pre if not np.isnan(t_offset) else np.nan      # total sys duration from Q-onset to T-offset (PEP + EP) -> QT
                subj_beh_ecg.at[index, columns['sys_dur_total']] = round(sys_dur_tot,3)
                sys_dur_pep = s_offset - q_onset_pre if not np.isnan(s_offset) else np.nan      # sys PEP duration from Q-onset to S-offset 
                subj_beh_ecg.at[index, columns['sys_dur_pep']] = round(sys_dur_pep,3)
                sys_dur_ep = t_offset - s_offset if not np.isnan(t_offset) else np.nan          # sys EP duration from S-offset to T-offset
                subj_beh_ecg.at[index, columns['sys_dur_ep']] = round(sys_dur_ep,3)
                
                ##### Diastole #####
                
                # Define diastole as interval from +50ms after systole end to Q-wave onset of subsequent R-peak
                dias_on = t_offset + 0.05 if not np.isnan(t_offset) else np.nan                 # add security buffer window of 50ms between sys and dia
                subj_beh_ecg.at[index, columns['dia_onset']] = round(dias_on,3)
                dias_dur = q_onset_post - dias_on if not np.isnan(q_onset_post) else np.nan     # total dia duration from +50ms after T-offset to Q-onset
                subj_beh_ecg.at[index, columns['dia_dur']] = round(dias_dur,3)             
                
                ##### Define in which cardiac phase the event falls #####

                # Check if event onset falls within systole (total interval) or diastole (total interval)
                subj_beh_ecg.at[index, columns[f'{abbrev}_sys']] = int(s_offset < event_time < t_offset)     # binary of event onset at systole
                subj_beh_ecg.at[index, columns[f'{abbrev}_dia']] = int(dias_on < event_time < q_onset_post)  # binary of event onset at diastole

                # Check if event onset falls within non-defined cardiac intervals (PEP, buffer window, post-diastole end)
                subj_beh_ecg.at[index, columns[f'{abbrev}_pep']] = int(rpeak_pre <= event_time <= s_offset)             # binary of event onset at systolic PEP (within R-R interval)
                subj_beh_ecg.at[index, columns[f'{abbrev}_buffer']] = int(t_offset <= event_time <= dias_on)            # binary of event onset at 50ms sys-dia buffer
                subj_beh_ecg.at[index, columns[f'{abbrev}_postdia']] = int(q_onset_post <= event_time <= rpeak_post)    # binary of event onset at post-dia window 
                

                ############## Segmentation into sys/dia bins of equal length ##############

                # If equalize_bins is True, calculate sys window from R-peak pre to T-offset, then use it to segment
                # dia window of equal duration starting from R-peak post backwards
                if equalize_bins:

                    sys_dur_equal = t_offset - rpeak_pre            # sys duration from R-peak pre to T-wave offset (will be = to dia duration)
                    subj_beh_ecg.at[index, columns['sys_dur_equal']] = round(sys_dur_equal,3)
                    dias_on_equal = rpeak_post - sys_dur_equal      # dia onset starts at point equal to sys duration backwards from R-peak post
                    subj_beh_ecg.at[index, columns['dia_onset_equal']] = round(dias_on_equal,3)
                    nondef_dur_equal = dias_on_equal - t_offset     # duration of non-defined interval between sys and dia
                    subj_beh_ecg.at[index, columns['nondef_dur_equal']] = round(nondef_dur_equal,3)

                    # Check if event onset falls within systole or diastole window of equal length
                    subj_beh_ecg.at[index, columns[f'{abbrev}_sys_equal']] = int(rpeak_pre < event_time < t_offset)        # binary of event onset at sys of equal length
                    subj_beh_ecg.at[index, columns[f'{abbrev}_dia_equal']] = int(dias_on_equal < event_time < rpeak_post)  # binary of event onset at dia of equal length


        beh_ecg_dfs.append(subj_beh_ecg)

    # Concatenate the dfs for all participants
    beh_ecg_df = pd.concat(beh_ecg_dfs, ignore_index=True)
    beh_ecg_df.rename(columns={'Unnamed: 0': 'subj_index'}, inplace=True)
    
    return beh_ecg_df

In [7]:
### Extract ECG features around action (keypress) onset ###

beh_ecg_action = extract_ecg_features_event(event_col=event_col, abbrev=abbrev)

In [8]:
### Extract ECG features around tone onset ###

beh_ecg_tone = extract_ecg_features_event(event_col=event_col_tone, abbrev=abbrev_tone)

## **4. (Optional) T-offset recomputation using Trapezium Area (TRA) approach**

If `recompute_toff_tra` is set to `True`, this section performs the recomputation of T-wave offsets on a trial-by-trial basis, using the Trapezium Area approach (TRA; Vázquez-Seisdedos et al., 2011). The TRA approach calculates successive areas of a rectangular trapezium with three fixed vertices (Xm, Ym, Xr) and one mobile vertex (Xi, Yi) in the downward slope of the T-wave, by shifting through the signal from (Xm, Ym) to (Xr, Yi). T-wave offset is defined as the x-coordinate of the moving point (Xi, Yi) that maximizes the trapezium area. In order:

- Load clean ECG signal, already band-pass filtered at 0.5-30Hz (4th order Butterworth), from the `_ecg-cleaned.tsv.gz` file included in `/derivatives/ecg-preproc/sub-xx/` (NB: to obtain this file, the ECG preprocessing pipeline `CardiAg_ecg_preproc.ipynb` should have been run with the variable `ecg_filter` set to `True`)
- Crop window that will include physiologically plausible T-wave, ranging from 75ms to 390ms after R-peak pre
- Extract T-peak position as local maximum within 3/4 of this physiologically plausible window
- Determine Xm point, having minimum value in 1st derivative (i.e., steepest downward slope), in 100ms window after T-peak
- Determine Xr point, having value closer to zero in 1st derivative, as reference point on isoelectric segment in 100ms window after Xm point
- Determine sequence of possible x- and y-coordinate between Xm (xm,ym) and Xr (xr,yr) where the mobile Xi point will be shifted through
- Determine T-wave offset as x-coordinate of mobile Xi point where the rectangular trapezium area (traced between the other fixed points and the mobile point) is maximal

Systole and diastole segmentation and durations are adjusted accordingly, using the TRA-based T-wave offset as end of systole (EP) and calculating diastole onset +50ms after T-wave offset, which is in turn used to determine the binary 0/1 binning of task events in each phase. All variables computed using the TRA approach are labelled with `_TRA` as an appendix. If the argument `equalize_bins` is `True`, an additional segmentation of systole and diastole is computed using windows of equal length (i.e., sys duration = dia duration), starting from R-peak pre and R-peak post respectively and using the TRA-based T-offset.The binary 0/1 binning of task events in each phase is also repeated for the equal length window approach. 

This section can be used to compare T-offset delineation performed by the two approaches, i.e., NeuroKit2 `ecg_delineate(method='dwt')` of the ECG pre-processing pipeline and TRA, which is especially useful in noisy signal where the former delineation may have failed. 

After recomputing the T-wave offset using the Trapezium Area approach, in the context of the CardiAg intentional binding task, the following steps are performed: 
- **4b. Replace selected participants with Trapezium Area (TRA) approach data**: replaces the T-wave offset and systole/diastole segmentation columns using the values from the TRA approach, for selected participants where the NeuroKit2 QRS delineation failed. Metadata about the T-wave offset detection method used is stored under the column 'toff_method'.
- **4c. Descriptives: individual cardiac phases duration**: calculate descriptives of the individual cardiac phases, including average systolic/diastolic duration for both action and tone trials, ranges of both systole and diastole intervals from R-peaks (including onset, offset and midpoint), and correlation of HR and systolic ejection period (EP) duration, also in proportion to the duration of the R-R interval. These are reported in Supplementary Results sections 1.2.1 and 1.2.2, in addition to Figures S1b-S1c and S2. 

In [9]:
############## 4a. (Optional) Trapezium Area approach for T-offset computation ##############

# Define function to recompute T-wave offset using Trapezium Area approach (Vázquez-Seisdedos et al., 2011)
# on a trial-by-trial basis
def compute_toff_tra(beh_ecg_df, event_col, abbrev, participant_col=column_map['participant'], 
                     sfreq=sfreq, equalize_bins=False):

    beh_ecg_toff = [] # initialize empty list to store dfs for all participants

    for subj in beh_ecg_df[participant_col].unique():

        print(f"----- Loading cleaned ECG for participant {subj} -----")

        # Load clean ECG (already band-pass filtered 0.5-30Hz, 4th order Butterworth)
        ecg_clean_fname = f'sub-{subj}_task-{exp_name}_ecg-cleaned.tsv.gz' 
        ecg_clean_dir = os.path.join(wd, 'derivatives', 'ecg-preproc', 
                                     f'sub-{subj}', datatype_name, ecg_clean_fname)

        physio_df = pd.read_csv(ecg_clean_dir, compression='gzip', sep='\t')
        ecg_cleaned = physio_df['ecg_cleaned'].values

        # Subset beh_ecg_df for that participant
        participant_trials = beh_ecg_df[beh_ecg_df[participant_col] == subj].copy()

        # Iterate over each trial
        for index, trial in participant_trials.iterrows():
            if pd.isna(trial[f'Rpeak_pre_{abbrev}']) or pd.isna(trial[f'Rpeak_post_{abbrev}']):
                continue        # Skip trial if R-peak data is missing (e.g., NoResp trials)

            # Extract positions of Rpeaks pre/post task event for each trial in idx
            rpeak_pre_idx = int(trial[f'Rpeak_pre_{abbrev}'] * sfreq)

            # Crop window for physiologically plausible T-wave (from 75ms to 390ms after Rpeak pre)
            t_window = ecg_cleaned[int(rpeak_pre_idx + 75):int(rpeak_pre_idx + 390)]

            # Extract T-peak position as local maximum within first 3/4 of specified window
            tpeak_idx = np.argmax(t_window[0:int(0.75*len(t_window))])  # get max value (T-peak) within first 3/4 of t_window
            tpeak_idx_glob = int(rpeak_pre_idx + 75) + tpeak_idx        # global idx of T-peak in ECG signal

            # Determine Xm point (min value in 1st derivative) in 100ms window after T-peak
            t_window_postmax = ecg_cleaned[int(tpeak_idx_glob):int(tpeak_idx_glob + 100)]
            xm = int(np.argmin(np.diff(t_window_postmax)))    # get min value of 1st derivative (i.e., T-wave highest negative gradient)
            ym = t_window_postmax[xm]                         # amplitude of Xm point
            xm_idx_glob = tpeak_idx_glob + xm                 # global idx of Xm point in ECG signal

            # Determine Xr point (isoelectric period) in 100ms window after Xm point 
            t_window_xr = ecg_cleaned[int(xm_idx_glob):int(xm_idx_glob + 100)]
            xr = np.argmin(np.abs(np.diff(t_window_xr)))      # get point closer to zero on 1st derivative
            xr_idx_glob = xm_idx_glob + xr                    # global idx of Xr point in ECG signal

            # Determine sequence of possible x- and y-coord for mobile point Xi between Xm and Xr
            xseq = np.arange(xm_idx_glob, xr_idx_glob)
            yseq = ecg_cleaned[xm_idx_glob:xr_idx_glob]

            # Define function to calculate trapezium area and select mobile point Xi that maximizes it 
            def trapez_area(xm, ym, xseq, yseq, xr):
                areas = [0.5 * (ym - yseq[i]) * ((2 * xr) - xseq[i] - xm) for i in range(len(xseq))]
                return np.argmax(areas)
            
            # Determine T-offset idx that maximizes trapezium area 
            toff = trapez_area(xm_idx_glob, ym, xseq, yseq, xr_idx_glob)
            xtoff = xm_idx_glob + toff  # x-coord of T-wave offset

            # Append timestamp of T-offset (in sec) into a new column
            toff_tra_s = xtoff / sfreq
            participant_trials.loc[index, f'Toffset_TRA_{abbrev}'] = toff_tra_s
            
            
            ##### Extraction of ECG features for binary analysis - TRA approach #####
            
            # Recalculate systole/diastole duration using T-offsets of TRA approach
            sys_dur_tot_tra = toff_tra_s - trial[f'Qonset_pre_{abbrev}']        # total sys duration from Q-onset to T-offset using TRA
            participant_trials.loc[index, f'sys_dur_total_TRA_{abbrev}'] = round(sys_dur_tot_tra,3) 
            sys_dur_ep_tra = toff_tra_s - trial[f'Soffset_{abbrev}']            # sys EP duration from S-offset to T-offset using TRA
            participant_trials.loc[index, f'sys_dur_ep_TRA_{abbrev}'] = round(sys_dur_ep_tra,3)    
            dias_on_tra = toff_tra_s + 0.05                                     # add security buffer window of 50ms between sys and dia from T-offset using TRA
            participant_trials.loc[index, f'dia_onset_TRA_{abbrev}'] = round(dias_on_tra,3)    
            dias_dur_tra = trial[f'Qonset_post_{abbrev}'] - toff_tra_s          # total dia duration from +50ms after T-offset using TRA to Q-onset
            participant_trials.loc[index, f'dia_dur_TRA_{abbrev}'] = round(dias_dur_tra,3) 

            ##### Define in which cardiac phase the event falls - TRA approach #####

            # Check if event onset falls within systole (total interval) or diastole (total interval)
            participant_trials.loc[index, f'{abbrev}_sys_TRA'] = int(trial[f'Soffset_{abbrev}'] < trial[event_col] < toff_tra_s)       # binary of event onset at systole
            participant_trials.loc[index, f'{abbrev}_dia_TRA'] = int(dias_on_tra < trial[event_col] < trial[f'Qonset_post_{abbrev}'])  # binary of event onset at diastole

            # Check if event onset falls within non-defined cardiac intervals (PEP, buffer window, post-diastole end)
            participant_trials.loc[index, f'{abbrev}_pep_TRA'] = int(trial[f'Rpeak_pre_{abbrev}'] <= trial[event_col] <= trial[f'Soffset_{abbrev}'])           # binary of event onset at systolic PEP (within R-R interval)
            participant_trials.loc[index, f'{abbrev}_buffer_TRA'] = int(toff_tra_s <= trial[event_col] <= dias_on_tra)                          # binary of event onset at 50ms sys-dia buffer
            participant_trials.loc[index, f'{abbrev}_postdia_TRA'] = (int(trial[f'Qonset_post_{abbrev}'] <= trial[event_col] <= trial[f'Rpeak_post_{abbrev}']))  # binary of event onset at post-dia window 


            ##### Segmentation into sys/dia bins of equal length - TRA approach #####
            
            if equalize_bins:
                sys_dur_equal = toff_tra_s - trial[f'Rpeak_pre_{abbrev}']        # sys duration from R-peak pre to T-wave offset
                participant_trials.loc[index, f'sys_dur_equal_TRA_{abbrev}'] = round(sys_dur_equal,3)
                dias_on_equal = trial[f'Rpeak_post_{abbrev}'] - sys_dur_equal       # dia onset starts at point equal to sys duration backwards from R-peak post
                participant_trials.loc[index, f'dia_onset_equal_TRA_{abbrev}'] = round(dias_on_equal,3)
                nondef_dur_equal = dias_on_equal - toff_tra_s                       # duration of non-defined interval between sys and dia
                participant_trials.loc[index, f'nondef_dur_equal_TRA_{abbrev}'] = round(nondef_dur_equal,3)

                # Check if event onset falls within systole or diastole, when of equal length
                participant_trials.loc[index, f'{abbrev}_sys_equal_TRA'] = int(trial[f'Rpeak_pre_{abbrev}'] < trial[event_col] < toff_tra_s)        # binary of event onset at systole of equal length
                participant_trials.loc[index, f'{abbrev}_dia_equal_TRA'] = int(dias_on_equal < trial[event_col] < trial[f'Rpeak_post_{abbrev}'])    # binary of event onset at diastole of equal length        
        
        beh_ecg_toff.append(participant_trials)


    # Concatenate the dfs for all participants
    beh_ecg_df_tra = pd.concat(beh_ecg_toff, ignore_index=True)  

    return beh_ecg_df_tra

In [ ]:
### Recompute T-wave offset for action onset trials using TRA approach (might take a few minutes) ###

if recompute_toff_tra: 
    beh_ecg_action_tra = compute_toff_tra(beh_ecg_df=beh_ecg_action, event_col=event_col, abbrev=abbrev)

In [ ]:
### Recompute T-wave offset for tone onset trials using TRA approach (might take a few minutes) ###

if recompute_toff_tra: 
    beh_ecg_tone_tra = compute_toff_tra(beh_ecg_df=beh_ecg_tone, event_col=event_col_tone, abbrev=abbrev_tone)

### **4b. Replace selected participants with Trapezium Area (TRA) approach data**

Replace the T-wave offset and systole/diastole segmentation columns using the values from the TRA approach, for selected participants where the NeuroKit2 QRS delineation failed. Metadata about the T-wave offset detection method used for a given participant is stored under the column 'toff_method'. 

The following participants were selected for the alternative TRA approach: [`sub-1`, `sub-21`, `sub-23`, `sub-25`, `sub-41`, `sub-51`, `sub-57`]

In [12]:
### Replace selected participants with TRA approach data ###

# Define function to substitute the T-wave offset delineated by NeuroKit2 with those by the TRA approach
# into one column, according to the specified participants
def replace_toff_data(beh_ecg_df_tra, failed_participants, trim_to_first_n_cols,
                      participant_col=column_map['participant'], abbrev=abbrev):
    """
    Replace the T-wave offset and systole/diastole segmentation columns using the values from the TRA approach, 
    for selected participants where the NeuroKit2 QRS delineation failed. Metadata about the T-wave offset 
    detection method used for a given participant is stored under the column 'toff_method'. 
    
    Parameters:
        -----------
        beh_ecg_df_tra : DataFrame
            DataFrame generated in the previous steps using the compute_toff_tra() function, containing
            preprocessed behavioral data and relevant ECG features around user-defined task-event, 
            computed both with 'neurokit' and 'tra' methods.  
        failed_participants : list
            List of participant IDs where NeuroKit2's T-offset detection failed.
        trim_to_first_n_cols : int or None 
            Number of columns to keep (from the beginning of beh_ecg_df_tra), usually excluding all 
            columns derived from compute_toff_tra() function. If None, keep all columns.
        participant_col : str
            Column name storing the participant IDs.   
        abbrev : str
            Abbreviated name for the event type (e.g., 'act'), used as suffix for all the derived columns. 

    Returns:
        --------
        DataFrame:
            DataFrame with the replaced T-offset and systole/diastole segmentation columns using the TRA-derived
            values only for the specified participants. 
    """

    # Columns to substitute
    tra_columns = [f'Toffset_TRA_{abbrev}', f'sys_dur_total_TRA_{abbrev}', f'sys_dur_ep_TRA_{abbrev}', 
                   f'dia_onset_TRA_{abbrev}', f'dia_dur_TRA_{abbrev}', f'{abbrev}_sys_TRA', 
                   f'{abbrev}_dia_TRA', f'{abbrev}_pep_TRA', f'{abbrev}_buffer_TRA', f'{abbrev}_postdia_TRA']
    
    original_columns = [col.replace('_TRA', '') for col in tra_columns]
    
    # Identify rows with failed T-offset detection participants
    is_failed = beh_ecg_df_tra[participant_col].isin(failed_participants)

    # Substitute the selected columns with TRA-derived values
    beh_ecg_df_fixed = beh_ecg_df_tra.copy()
    beh_ecg_df_fixed.loc[is_failed, original_columns] = beh_ecg_df_tra.loc[is_failed, tra_columns].values

    # Add method label
    beh_ecg_df_fixed['toff_method'] = 'neurokit'
    beh_ecg_df_fixed.loc[is_failed, 'toff_method'] = 'tra'

    # Trim columns if requested
    if trim_to_first_n_cols is not None:
        cols_to_keep = list(beh_ecg_df_fixed.columns[:trim_to_first_n_cols]) + ['toff_method']
        return beh_ecg_df_fixed[cols_to_keep]
    else:
        return beh_ecg_df_fixed

In [13]:
# Define list of participants where T-wave offset detection using NeuroKit2 failed
failed_toff_subj = [1, 21, 23, 25, 41, 51, 57]

# Replace T-wave offset and sys/dia segmentation for action onset trials on selected participants
beh_ecg_action_clean = replace_toff_data(beh_ecg_df_tra=beh_ecg_action_tra, failed_participants=failed_toff_subj, 
                                         trim_to_first_n_cols=len(beh_ecg_action.columns), abbrev=abbrev)

# Replace T-wave offset and sys/dia segmentation for tone onset trials on selected participants
beh_ecg_tone_clean = replace_toff_data(beh_ecg_df_tra=beh_ecg_tone_tra, failed_participants=failed_toff_subj, 
                                       trim_to_first_n_cols=len(beh_ecg_tone.columns), abbrev=abbrev_tone)

In [15]:
# Save output files as TSV in 'derivatives/cardiac-phase-analysis' directory

beh_ecg_action_dir = os.path.join(cardphase_dir, 'task-CardiAgIBTask_beh_ecg_action.tsv')
beh_ecg_action_clean.to_csv(beh_ecg_action_dir, index=False, sep='\t', na_rep="n/a")

beh_ecg_tone_dir = os.path.join(cardphase_dir, 'task-CardiAgIBTask_beh_ecg_tone.tsv')
beh_ecg_tone_clean.to_csv(beh_ecg_tone_dir, index=False, sep='\t', na_rep="n/a")

### **4c. Descriptives: individual cardiac phases duration**

This section provides descriptives of the individual cardiac phases, including average systolic/diastolic duration for both action and tone trials, ranges of both systole and diastole intervals from R-peaks (including onset, offset and midpoint), and correlation of HR and systolic ejection period (EP) duration, also in proportion to the duration of the R-R interval. These are reported in Supplementary Results sections 1.2.1 and 1.2.2. In addition, the following plots are created:

- Density histogram of mean systolic and diastolic durations (Figure S1b)
- Range plot of individual systolic/diastolic intervals from R-peak with SD and midpoint (Figure S1c)
- Correlation plot of HR and systolic EP duration (Figure S2)

In [ ]:
# Create directory for saving plots
project_root = os.path.dirname(wd)
plotting_dir = os.path.join(project_root, 'results', 'datavisualization')
if not os.path.exists(plotting_dir):
    os.makedirs(plotting_dir)

In [39]:
### Descriptives of individual cardiac phases - mean phase duration ###

# Systole
sys_dur_ep_act = beh_ecg_action_clean[['subjID', 'sys_dur_ep_act']].rename(columns={'sys_dur_ep_act': 'sys_duration'})
sys_dur_ep_tone = beh_ecg_tone_clean[['subjID', 'sys_dur_ep_tone']].rename(columns={'sys_dur_ep_tone': 'sys_duration'})
sys_dur_ep_overall = pd.concat([sys_dur_ep_act, sys_dur_ep_tone], axis=0, join='outer')
sys_dur_mean = (sys_dur_ep_overall['sys_duration'].mean()) * 1000
sys_dur_sd = (sys_dur_ep_overall['sys_duration'].std()) * 1000
sys_dur_min = (sys_dur_ep_overall.groupby('subjID')['sys_duration'].min()).mean() * 1000
sys_dur_max = (sys_dur_ep_overall.groupby('subjID')['sys_duration'].max()).mean() * 1000
 
print(f"Mean systole duration: {round(sys_dur_mean, 2)} +/- {round(sys_dur_sd, 2)} ms")
print(f"Systole range: {round(sys_dur_min)} - {round(sys_dur_max)} ms")

# Diastole
dia_dur_act = beh_ecg_action_clean[['subjID', 'dia_dur_act']].rename(columns={'dia_dur_act': 'dia_duration'})
dia_dur_tone = beh_ecg_tone_clean[['subjID', 'dia_dur_tone']].rename(columns={'dia_dur_tone': 'dia_duration'})
dia_dur_overall = pd.concat([dia_dur_act, dia_dur_tone], axis=0, join='outer')
dia_dur_mean = (dia_dur_overall['dia_duration'].mean()) * 1000
dia_dur_sd = (dia_dur_overall['dia_duration'].std()) * 1000
dia_dur_min = (dia_dur_overall.groupby('subjID')['dia_duration'].min()).mean() * 1000
dia_dur_max = (dia_dur_overall.groupby('subjID')['dia_duration'].max()).mean() * 1000
 
print(f"Mean diastole duration: {round(dia_dur_mean, 2)} +/- {round(dia_dur_sd, 2)} ms")
print(f"Diastole range: {round(dia_dur_min)} - {round(dia_dur_max)} ms")

# Compute per-subject mean durations
systole_means = sys_dur_ep_overall.groupby('subjID')['sys_duration'].mean().reset_index()
systole_means['Cardiac Phase'] = 'Systole'
systole_means = systole_means.rename(columns={'sys_duration': 'Duration'})
diastole_means = dia_dur_overall.groupby('subjID')['dia_duration'].mean().reset_index()
diastole_means['Cardiac Phase'] = 'Diastole'
diastole_means = diastole_means.rename(columns={'dia_duration': 'Duration'})

# Combine mean systolic and diastolic durations in a long DataFrame
sysdia_means_long = pd.concat([systole_means, diastole_means], ignore_index=True)

Mean systole duration: 273.71 +/- 29.91 ms
Systole range: 232 - 297 ms
Mean diastole duration: 441.07 +/- 150.88 ms
Diastole range: 266 - 652 ms


In [ ]:
### FIGURE S1b ###
# Density histogram of mean systolic and diastolic durations per participant 
plt.figure(figsize=(6, 6), dpi=300)
sysdia_colors = ['#ff7f00', '#7AC5CD']
sns.histplot(data=sysdia_means_long, x='Duration', hue='Cardiac Phase', palette=sysdia_colors, bins=30, 
             element='bars', stat='count', common_norm=False, alpha=0.7, edgecolor='black')
plt.xlabel('Mean phase duration over participants (s)', fontsize='x-large')
plt.ylabel('Number of participants', fontsize='x-large')
sns.despine()
plt.tight_layout()

# Save plot 
figS1_1 = os.path.join(plotting_dir, "FigureS1b_sysdia_durations.svg")
plt.savefig(figS1_1, format="svg")
plt.show()

In [ ]:
### Descriptives of individual cardiac phases - systolic/diastolic ranges from R-peak ###

# Define common ECG columns with their suffixes
act_cols = ['Rpeak_pre_act', 'Soffset_act', 'Toffset_act', 'dia_onset_act', 'Qonset_post_act', 'HR_1perMin_act', 'RR_s_act']
tone_cols = ['Rpeak_pre_tone', 'Soffset_tone', 'Toffset_tone', 'dia_onset_tone', 'Qonset_post_tone', 'HR_1perMin_tone', 'RR_s_tone']

# Remove suffixes from action/tone df and concatenate them
act_df = beh_ecg_action_clean[['subjID'] + act_cols].rename(columns={col: col.replace('_act', '') for col in act_cols})
tone_df = beh_ecg_tone_clean[['subjID'] + tone_cols].rename(columns={col: col.replace('_tone', '') for col in tone_cols})
cardphase_onoff = pd.concat([act_df, tone_df], ignore_index=True)
cardphase_onoff = cardphase_onoff.dropna(axis='index')

# Compute onset and offset of systole/diastole in relation to previous R-peak
cardphase_onoff['sys_onset_s'] = cardphase_onoff['Soffset'] - cardphase_onoff['Rpeak_pre']
cardphase_onoff['sys_offset_s'] = cardphase_onoff['Toffset'] - cardphase_onoff['Rpeak_pre']
cardphase_onoff['sys_dur'] = cardphase_onoff['sys_offset_s'] - cardphase_onoff['sys_onset_s']

cardphase_onoff['dia_onset_s'] = cardphase_onoff['dia_onset'] - cardphase_onoff['Rpeak_pre']
cardphase_onoff['dia_offset_s'] = cardphase_onoff['Qonset_post'] - cardphase_onoff['Rpeak_pre']
cardphase_onoff['dia_dur'] = cardphase_onoff['dia_offset_s'] - cardphase_onoff['dia_onset_s']

# Compute mean onset and offset per participant and phase
cardphase_onoff_means = cardphase_onoff.groupby('subjID').agg({
    'sys_onset_s': 'mean', 'sys_offset_s': 'mean',
    'dia_onset_s': 'mean', 'dia_offset_s': 'mean', 
    'sys_dur': ['mean', 'std'], 'dia_dur':  ['mean', 'std']
}).reset_index()
cardphase_onoff_means.columns = [
    f"{col[0]}" if col[1] == '' else f"{col[0]}_{col[1]}" for col in cardphase_onoff_means.columns]

# Compute midpoint for plotting
cardphase_onoff_means['sys_mid'] = (cardphase_onoff_means['sys_onset_s_mean'] + cardphase_onoff_means['sys_offset_s_mean']) / 2
cardphase_onoff_means['dia_mid'] = (cardphase_onoff_means['dia_onset_s_mean'] + cardphase_onoff_means['dia_offset_s_mean']) / 2

# Sort participants by mean diastolic duration
cardphase_onoff_means = cardphase_onoff_means.sort_values('dia_dur_mean', ascending=True).reset_index(drop=True)

# Print ranges of systole/diastole onset and offset
print("----- Systole -----")
print(f"Mean systole onset: {round((cardphase_onoff_means['sys_onset_s_mean'].mean() * 1000),2)} +/- {round((cardphase_onoff_means['sys_onset_s_mean'].std() * 1000),2)} ms")
print(f"Systole onset ranges: {round((cardphase_onoff_means['sys_onset_s_mean'].min() * 1000))} - {round((cardphase_onoff_means['sys_onset_s_mean'].max() * 1000))} ms")
print(f"Mean systole offset: {round((cardphase_onoff_means['sys_offset_s_mean'].mean() * 1000),2)} +/- {round((cardphase_onoff_means['sys_offset_s_mean'].std() * 1000),2)} ms")
print(f"Systole offset ranges: {round((cardphase_onoff_means['sys_offset_s_mean'].min() * 1000))} - {round((cardphase_onoff_means['sys_offset_s_mean'].max() * 1000))} ms")

print("\n----- Diastole -----")
print(f"Mean diastole onset: {round((cardphase_onoff_means['dia_onset_s_mean'].mean() * 1000),2)} +/- {round((cardphase_onoff_means['dia_onset_s_mean'].std() * 1000),2)} ms")
print(f"Diastole onset ranges: {round((cardphase_onoff_means['dia_onset_s_mean'].min() * 1000))} - {round((cardphase_onoff_means['dia_onset_s_mean'].max() * 1000))} ms")
print(f"Mean diastole offset: {round((cardphase_onoff_means['dia_offset_s_mean'].mean() * 1000),2)} +/- {round((cardphase_onoff_means['dia_offset_s_mean'].std() * 1000),2)} ms")
print(f"Diastole offset ranges: {round((cardphase_onoff_means['dia_offset_s_mean'].min() * 1000))} - {round((cardphase_onoff_means['dia_offset_s_mean'].max() * 1000))} ms")


### FIGURE S1c ###

# Create plotting DataFrame
plot_df = pd.DataFrame({
    'subjID': pd.concat([cardphase_onoff_means['subjID'], cardphase_onoff_means['subjID']]),
    'Phase': ['Systole'] * len(cardphase_onoff_means) + ['Diastole'] * len(cardphase_onoff_means),
    'Mid': pd.concat([cardphase_onoff_means['sys_mid'], cardphase_onoff_means['dia_mid']]),
    'Duration': pd.concat([cardphase_onoff_means['sys_dur_mean'], cardphase_onoff_means['dia_dur_mean']]), 
    'SD': pd.concat([cardphase_onoff_means['sys_dur_std'], cardphase_onoff_means['dia_dur_std']])
})
plot_df['subjID'] = pd.Categorical(plot_df['subjID'], categories=cardphase_onoff_means['subjID'], ordered=True)
plot_df['y_pos'] = plot_df['subjID'].cat.codes

# Range plot of individual systolic/diastolic intervals with SD and midpoint
fig, ax = plt.subplots(figsize=(6, 6), dpi=300)
colors = {'Systole': '#ff7f00', 'Diastole': '#7AC5CD'}

for _, row in plot_df.iterrows():
    start = row['Mid'] - row['Duration'] / 2
    y = row['y_pos']
    end = start + row['Duration']

    # Mean phase duration spanning onset/offset from preceding R-peak (colored bar)
    ax.barh(y=y, width=row['Duration'], left=start, height=0.6,
            color=colors[row['Phase']], edgecolor='white', alpha=0.8)

    # Midpoint of cardiac phase (black dot)
    ax.plot(row['Mid'], y, 'ko', markersize=3, zorder=5)  # 'ko' means black circle

    # Standard deviation of phase duration (black line)
    ax.hlines(y, start - row['SD'], start, color='black', linewidth=1)
    ax.hlines(y, end, end + row['SD'], color='black', linewidth=1)

ax.set_xlabel('Distance to previous R-peak (s)', fontsize='x-large')
ax.set_ylabel('Participants (sorted by diastole duration)', fontsize='x-large')

# Manual legend
handles = [plt.Rectangle((0, 0), 1, 1, color=colors[phase]) for phase in colors]
labels = list(colors.keys())
ax.legend(handles, labels, title='Cardiac Phase')

sns.despine()
plt.xlim(0, 1.5)
plt.tight_layout()

# Save plot 
figS1_2 = os.path.join(plotting_dir, "FigureS1c_sysdia_ranges.svg")
plt.savefig(figS1_2, format="svg", bbox_inches='tight')
plt.show()

In [ ]:
### Descriptives of individual cardiac phases - correlation of HR and systolic EP duration ###

# Merge df with subject-level average systolic EP duration, heart rate and proportion of systolic EP to RR
sys_dur_ep_subj = sys_dur_ep_overall.groupby('subjID').mean()
cardphase_onoff_subjmeans = cardphase_onoff.groupby('subjID').mean()
sys_dur_hr_df = sys_dur_ep_subj.join(cardphase_onoff_subjmeans[['HR_1perMin', 'RR_s']], how='left')
sys_dur_hr_df['prop_sys2rr'] = sys_dur_hr_df['sys_duration'] / sys_dur_hr_df['RR_s']

# Compute correlation of mean HR and duration systolic EP
sys_dur_hr_corr = pg.corr(sys_dur_hr_df['HR_1perMin'], sys_dur_hr_df['sys_duration'], alternative='two-sided')
print(sys_dur_hr_corr)

# Compute correlation of mean HR and proportion of systolic EP to RR interval duration
prop_sys2rr_hr_corr = pg.corr(sys_dur_hr_df['HR_1perMin'], sys_dur_hr_df['prop_sys2rr'], alternative='two-sided')
print(prop_sys2rr_hr_corr)

# Print summary of systolic EP ranges
print(f"\nSystolic EP ranges: {round(sys_dur_hr_df['sys_duration'].min(),2)} - {round(sys_dur_hr_df['sys_duration'].max(),2)} s (M = {round(sys_dur_hr_df['sys_duration'].mean(),2)}, SD = {round(sys_dur_hr_df['sys_duration'].std(),2)})")
print(f"Systolic EP proportion: {round(sys_dur_hr_df['prop_sys2rr'].min(),2)} - {round(sys_dur_hr_df['prop_sys2rr'].max(),2)} (M = {round(sys_dur_hr_df['prop_sys2rr'].mean(),2)}, SD = {round(sys_dur_hr_df['prop_sys2rr'].std(),2)})")

### FIGURE S2 ###
# Correlation plot of HR and systolic EP duration
fig, axs = plt.subplots(1, 2, figsize=(10, 5), dpi=300)

sns.regplot(data=sys_dur_hr_df, x='HR_1perMin', y='sys_duration', ax=axs[0],
            marker='s', scatter_kws={'color': 'black', 'alpha': 0.7}, line_kws={'color': 'black'})
axs[0].set_xlabel("Heart Rate (bpm)", fontsize='x-large')
axs[0].set_ylabel("Systolic Ejection Phase (EP) (sec)", fontsize='x-large')
#axs[0].set_title("Correlation of Systolic EP and Instantaneous HR")
sns.despine(ax=axs[0])

sns.regplot(
    data=sys_dur_hr_df, x='HR_1perMin', y='prop_sys2rr', ax=axs[1],
    marker='s', scatter_kws={'color': 'black', 'alpha': 0.7}, line_kws={'color': 'black'}
)
axs[1].set_xlabel("Heart Rate (bpm)", fontsize='x-large')
axs[1].set_ylabel("Proportion of Systolic EP rel. to R-R", fontsize='x-large')
#axs[1].set_title("Correlation of Systolic EP Proportion and Instantaneous HR")
sns.despine(ax=axs[1])
plt.tight_layout()

# Save plot 
figS2 = os.path.join(plotting_dir, "FigureS2_sysEP_heartrate.svg")
plt.savefig(figS2, format="svg", bbox_inches='tight')
plt.show()

## **5. (Optional) Individual plot of ECG around task event with sys/dia segmentation**

If `plot_ecg2event` is set to `True`, an interactive plot showing the individual ECG trace for a user-defined participant with task event onset (on each trial, condition and - optionally - block) is created. The interactive plot displays:

- On top left, buttons and drop-down menu to select condition, block (optional) and trial for the user-defined participant (the column names for participant, trial, condition and block can be supplied as a dict usig the `column_map` method)
- Shadowed areas corresponding to systole (orange) and diastole (blue) intervals, according to the specified method between NeuroKit2 'DWT' approach (default) or the re-computed TRA approach (using the `toff_method` argument) 
- A red X mark corresponding to the T-wave offset using the alternative approach
- A vertical dotted line showing the time of the user-defined task event (the displayed task event label can be specified using the argument `event_label`, e.g. 'Action onset')
- On bottom, an horizontal arrow showing the duration of the R-R interval

In [ ]:
############## 5. (Optional) individual plot of ECG around task event with sys/dia segmentation ##############

if plot_ecg2event: 

    # Plot (if available, cleaned) ECG signal for one participant around the user-defined task event, 
    # with colored intervals for sys/dia, task event onset, RR interval duration
    cardphase_utils.plot_individual_ecg_by_event(behpreproc_ecg_long=beh_ecg_action_tra,  
                                                 participant_id = 27,             # change with subj ID of your choice
                                                 column_map=column_map, toff_method='nk', sfreq=sfreq,
                                                 event_col='keypress_onset', abbrev='act', event_label='Action onset',  
                                                 wd=wd, exp_name=exp_name, datatype_name=datatype_name)

## **6. Binary analysis - systolic vs. diastolic ratio of task event [Results 2.2.2]**

This section performs binary analysis of individual cardiac phases (i.e., systole vs. diastole) with respect to a task event of interest (user-defined). To take into account between-subject differences in HR and cardiac phase lengths, the phase-specific proportion of task events, relative to all trials, is normalized by the proportion of the respective cardiac phase in the entire RR interval. Lastly, in group-level analysis, normalized systolic and diastolic ratios are tested using a two-sided paired t-test. If no cardiac effect is present, task events would be randomly distributed across both cardiac phases, and the task event ratio (N events/N trials) should correspond to the cardiac phase ratio (phase length/whole cycle), i.e., ratio value = 1. If there is a cardiac effect, the phase-specific ratio value should be > 1, indicating over-proportional accumulation of task events in the respective phase. 

In detail, this section includes: 

- **6a. Define normalized ratio for both phases (systole, diastole)**: the sum of task events (e.g., keypress or stimulus onset) per each phase (as ratio of all trials that include the task event) is normalized to the proportion of the subject-specific phase length in the total cardiac cycle. The systolic or diastolic ratio is thus derived from: (N events per phase / total N trials) / (individual phase length/ individual mean R-R length). The user can specify whether to use the systole/diastole segmentation produced using NeuroKit2 'DWT' method (default) or the alternative Trapezium Area 'TRA' approach. 
- **6b. Run group-level binary analysis on systolic vs. diastolic ratio**: at the group level, the normalized systolic and diastolic ratios for the task event are tested against each other using a two-sided paired t-test. This is reported in Results section 2.2.2. 
- **6c. Scatter plot for systolic vs. diastolic ratio**: create a scatterplot showing systolic vs. diastolic ratio for each participant, with dashed lines tracing the expected phase-specific ratio of task event onsets if uniformly distributed (i.e., ratio = 1). Instead, orange markers indicate participants with an over-proportional accumulation of task events in systole (systolic ratio >1, diastolic <1), blue markers indicate participants with an over-proportional accumulation of task events in diastole (diastolic ratio >1, systolic <1), whilst grey markers indicate participants with no phase preference. This corresponds to Figure 3b. 
- **6d. Condition-wise analysis of systolic vs. diastolic ratio**: at the group level, separately per condition of the intentional binding task, the normalized systolic and diastolic ratios for either action or tone trials are tested against each other using a a two-sided paired t-test. This is also reported in Results section 2.2.2. 

### **6a. Define normalized ratio for both phases (systole, diastole)**

In [18]:
############## 6a. Define normalized ratio for both phases (systole, diastole) ##############

# Define a custom function for computing the normalized systolic and diastolic ratios: 
# (N events per phase / total N trials) / (individual mean phase length/ individual mean R-R length)

def analyse_sysdia_ratio(beh_ecg_df, abbrev, participant_col=column_map['participant'], toff_method='neurokit'):
    """
    Compute the normalized systolic and diastolic ratios for cardiac phase binary analysis. 

    Parameters:
        -----------
        beh_ecg_df : DataFrame
            DataFrame generated in the previous steps using the extract_ecg_features_event() function, containing
            preprocessed behavioral data and relevant ECG features around user-defined task-event. 
        abbrev : str
            Abbreviated name for the event type (e.g., 'act'), used as suffix for all the derived columns. 
            Should be the same as the one used in the extract_ecg_features_event() function to generate beh_ecg_df. 
        participant_col : str
            Column name storing the participant IDs.   
        toff_method : str, default 'neurokit'
            Method to use for T-wave offset delineation and sys/dia segmentation, either 'neurokit' (default, 
            refers to NeuroKit2 'DWT' method) or 'tra' (Trapezium Area approach, see S4). 

    Returns:
        --------
        DataFrame:
            DataFrame with the wide-format participant data used for cardiac phase binary analysis, 
            incl. count of task events per each phase (sys, dia, non-defined) normalized by phase proportion. 

    """
    
    # Specify the task events columns per each phase based on selected method (NeuroKit2 or TRA)
    # Systole only including EP, non-defined intervals including PEP, 50ms buffer window & post-diastole window
    if toff_method.lower() in ['neurokit', 'neurokit2', 'nk', 'dwt']:        
        sys_bin = f'{abbrev}_sys' 
        dia_bin = f'{abbrev}_dia'
        pep_bin = f'{abbrev}_pep'
        buffer_bin = f'{abbrev}_buffer'
        postdia_bin = f'{abbrev}_postdia'
        nondef_bin = f'{abbrev}_nondef'
        sys_dur_ep = f'sys_dur_ep_{abbrev}'
        dia_dur = f'dia_dur_{abbrev}'

    elif toff_method.lower() in ['tra', 'trapezium']:
        sys_bin = f'{abbrev}_sys_TRA'
        dia_bin = f'{abbrev}_dia_TRA'
        pep_bin = f'{abbrev}_pep_TRA'
        buffer_bin = f'{abbrev}_buffer_TRA'
        postdia_bin = f'{abbrev}_postdia_TRA'
        nondef_bin = f'{abbrev}_nondef_TRA'
        sys_dur_ep = f'sys_dur_ep_TRA_{abbrev}'
        dia_dur = f'dia_dur_TRA_{abbrev}'
    else:
        raise ValueError("Wrong method! T-offset delineation method should be either 'NeuroKit' or 'TRA'.")
    
    # Summarize the count of task events per each phase (systole, diastole & non-defined intervals), based on selected method
    event_phase_ratio_df = beh_ecg_df.pivot_table(index=participant_col, values=[sys_bin, dia_bin, pep_bin, buffer_bin, postdia_bin],
                                                                                 aggfunc=lambda x: (x == 1).sum())
    
    # Merge the non-defined phase columns into one and drop the original ones
    event_phase_ratio_df[nondef_bin] = (event_phase_ratio_df[pep_bin] +         # systolic PEP from R-peak to S-offset
                                        event_phase_ratio_df[buffer_bin] +      # 50ms buffer after T-offset
                                        event_phase_ratio_df[postdia_bin])      # post-dia from R-onset to next R-peak
    event_phase_ratio_df.drop(columns=[pep_bin, buffer_bin, postdia_bin], inplace=True)

    # Compute total number of trials that include task event across cardiac and non-defined intervals
    # NB: if N trials is fixed for all conditions (with no exclusions), it should equal the expected N trials from your design
    event_phase_ratio_df[f'{abbrev}_tot_ntrials'] = (event_phase_ratio_df[sys_bin] + event_phase_ratio_df[dia_bin] + 
                                                     event_phase_ratio_df[nondef_bin])
    
    # Compute the participant-specific average of RR intervals and sys/dia durations
    event_phase_rr = beh_ecg_df.pivot_table(index=participant_col, 
                                            values=[f'RR_s_{abbrev}', sys_dur_ep, dia_dur])
    event_phase_ratio_df = event_phase_ratio_df.join(event_phase_rr) # join the summary tables

    # Compute normalized systolic and diastolic ratio
    # (N events per phase / total N trials) / (individual mean phase length/ individual mean R-R length)
    event_phase_ratio_df[f'{abbrev}_sys_ratio'] = (
        (event_phase_ratio_df[sys_bin] / event_phase_ratio_df[f'{abbrev}_tot_ntrials']) / 
        (event_phase_ratio_df[sys_dur_ep] / event_phase_ratio_df[f'RR_s_{abbrev}']))

    event_phase_ratio_df[f'{abbrev}_dia_ratio'] = (
        (event_phase_ratio_df[dia_bin] / event_phase_ratio_df[f'{abbrev}_tot_ntrials']) / 
        (event_phase_ratio_df[dia_dur] / event_phase_ratio_df[f'RR_s_{abbrev}']))
    
    # Compute proportion of cardiac phase intervals relative to total cardiac cycle
    # (individual mean phase length/ individual mean R-R length)
    event_phase_ratio_df[f'{abbrev}_sys_prop2rr'] = event_phase_ratio_df[sys_dur_ep] / event_phase_ratio_df[f'RR_s_{abbrev}']
    event_phase_ratio_df[f'{abbrev}_dia_prop2rr'] = event_phase_ratio_df[dia_dur] / event_phase_ratio_df[f'RR_s_{abbrev}']

    # Re-organize column order and reset participant ID index 
    event_phase_ratio_df = event_phase_ratio_df.reset_index()
    col_order = [participant_col, sys_bin, dia_bin, nondef_bin, f'{abbrev}_tot_ntrials', 
                 f'RR_s_{abbrev}', sys_dur_ep, dia_dur, f'{abbrev}_sys_ratio', 
                 f'{abbrev}_dia_ratio', f'{abbrev}_sys_prop2rr', f'{abbrev}_dia_prop2rr']
    event_phase_ratio_df = event_phase_ratio_df[col_order]


    return event_phase_ratio_df

In [19]:
### Calculate normalized systolic & diastolic ratios for action onset ###

act_sysdia_ratio = analyse_sysdia_ratio(beh_ecg_df=beh_ecg_action_clean, abbrev=abbrev)

In [20]:
### Calculate normalized systolic & diastolic ratios for tone onset ###

tone_sysdia_ratio = analyse_sysdia_ratio(beh_ecg_df=beh_ecg_tone_clean, abbrev=abbrev_tone)

### **6b. Run group-level binary analysis on systolic vs. diastolic ratio**

In [24]:
############## 6b. Run group-level binary analysis on systolic vs. diastolic ratio ##############

# Group-level binary analysis of action onset: two-sided paired t-test of sys vs. dia ratio
sysdia_ratio_ttest = pg.ttest(act_sysdia_ratio[f'{abbrev}_sys_ratio'], act_sysdia_ratio[f'{abbrev}_dia_ratio'], 
                               paired=True, alternative='two-sided')

# Summary of group-level results: action trials
md("Binary analysis showed a significantly larger (t({})={}, p={}, Cohen's D={}) ratio of action onsets in systole (M={}, SD={}) as compared to diastole (M={}, SD={}).".format(
    sysdia_ratio_ttest['dof'].iloc[0], round(sysdia_ratio_ttest['T'].iloc[0],3), 
    round(sysdia_ratio_ttest['p_val'].iloc[0],5), round(sysdia_ratio_ttest['cohen_d'].iloc[0],3),
    round(act_sysdia_ratio[f'{abbrev}_sys_ratio'].mean(),2), round(act_sysdia_ratio[f'{abbrev}_sys_ratio'].std(),2), 
    round(act_sysdia_ratio[f'{abbrev}_dia_ratio'].mean(),2), round(act_sysdia_ratio[f'{abbrev}_dia_ratio'].std(),2)
))

Binary analysis showed a significantly larger (t(43)=3.134, p=0.00311, Cohen's D=0.772) ratio of action onsets in systole (M=1.02, SD=0.1) as compared to diastole (M=0.95, SD=0.08).

In [25]:
# Group-level binary analysis of tone onset: two-sided paired t-test of sys vs. dia ratio
tone_sysdia_ratio_ttest = pg.ttest(tone_sysdia_ratio['tone_sys_ratio'], tone_sysdia_ratio['tone_dia_ratio'], 
                                   paired=True, alternative='two-sided')
                                   
# Summary of group-level results: tone trials
md("Binary analysis showed a marginally significant larger (t({})={}, p={}, Cohen's D={}) ratio of tone onsets in systole (M={}, SD={}) as compared to diastole (M={}, SD={}).".format(
    tone_sysdia_ratio_ttest['dof'].iloc[0], round(tone_sysdia_ratio_ttest['T'].iloc[0],3), 
    round(tone_sysdia_ratio_ttest['p_val'].iloc[0],5), round(tone_sysdia_ratio_ttest['cohen_d'].iloc[0],3),
    round(tone_sysdia_ratio[f'tone_sys_ratio'].mean(),2), round(tone_sysdia_ratio[f'tone_sys_ratio'].std(),2), 
    round(tone_sysdia_ratio[f'tone_dia_ratio'].mean(),2), round(tone_sysdia_ratio[f'tone_dia_ratio'].std(),2)
))

Binary analysis showed a marginally significant larger (t(43)=2.104, p=0.04129, Cohen's D=0.53) ratio of tone onsets in systole (M=1.02, SD=0.12) as compared to diastole (M=0.96, SD=0.08).

###  **6c. Scatter plot for systolic vs. diastolic ratio**

In [26]:
############## 6c. Scatter plot for systolic vs. diastolic ratio ##############

def plot_sysdia_ratio(event_phase_ratio_df, abbrev, plot_fname, 
                      participant_col=column_map['participant'], toff_method='neurokit', xy_lim=[0.7,1.3]):
    """
    Create scatterplot showing systolic vs. diastolic ratio for each participant. 
    Dashed lines indicate expected phase-specific ratio of task event if uniformly distributed (i.e., ratio = 1). 
    Orange markers indicate systolic preference (systolic ratio >1, diastolic <1), blue markers indicate 
    diastolic preference (diastolic ratio >1, systolic <1), grey markers indicate no phase preference. 

    Parameters:
        -----------
        event_phase_ratio_df : DataFrame
            DataFrame generated in the previous steps using the analyse_sysdia_ratio() function, containing
            preprocessed behavioral data and relevant ECG features around user-defined task-event. 
        abbrev : str
            Abbreviated name for the event type (e.g., 'act'), used as suffix for all the derived columns. Should be the same
            as the one used in the extract_ecg_features_event() function to generate beh_ecg_df. 
        participant_col : str
            Column name storing the participant IDs. 
        toff_method: str, default 'neurokit'
            Method to use for T-wave offset delineation and sys/dia segmentation, either 'neurokit' (default, 
            refers to NeuroKit2 'DWT' method) or 'tra' (Trapezium Area approach, see S4). 
            
    Returns:
        --------
        Scatterplot. 

    """

    # Define column names based on chosen T-wave offset delineation method
    if toff_method.lower() in ['neurokit', 'neurokit2', 'nk', 'dwt']:
        sys_bin = f'{abbrev}_sys' 
        dia_bin = f'{abbrev}_dia'

    elif toff_method.lower() in ['tra', 'trapezium']:
        sys_bin = f'{abbrev}_sys_TRA'
        dia_bin = f'{abbrev}_dia_TRA'
    else:
        raise ValueError("Wrong method! T-offset delineation method should be either 'NeuroKit' or 'TRA'.")

    # Melt the event_phase_ratio_df to long format
    event_phase_ratio_df_long = event_phase_ratio_df.melt(id_vars=participant_col,
                                                          value_vars=[f'{abbrev}_sys_ratio', f'{abbrev}_dia_ratio'],
                                                          var_name='phase', value_name=f'{abbrev}_relratio')

    # Aggregate summary of mean, SD and count per cardiac phase ratio
    event_phase_ratio_agg = event_phase_ratio_df_long.groupby('phase').agg(
        mean=(f'{abbrev}_relratio', 'mean'),
        sd=(f'{abbrev}_relratio', 'std'),
        count=(f'{abbrev}_relratio', 'count')).reset_index()

    # Compute standard error and confidence intervals
    event_phase_ratio_agg['se'] = event_phase_ratio_agg['sd'] / np.sqrt(event_phase_ratio_agg['count'])
    event_phase_ratio_agg['se_up'] = event_phase_ratio_agg['mean'] + event_phase_ratio_agg['se']
    event_phase_ratio_agg['se_down'] = event_phase_ratio_agg['mean'] - event_phase_ratio_agg['se']

    # Compute differences between task event rate (i.e., N events per phase to N trials) and phase rate (i.e., phase length to RR)
    event_phase_ratio_df[f'{abbrev}_diff_ratio2sys'] = ((event_phase_ratio_df[sys_bin] / event_phase_ratio_df[f'{abbrev}_tot_ntrials']) - 
                                                       event_phase_ratio_df[f'{abbrev}_sys_prop2rr']) # event rate sys - proportion sys
    event_phase_ratio_df[f'{abbrev}_check_ratio2sys'] = np.sign(event_phase_ratio_df[f'{abbrev}_diff_ratio2sys']) # check whether event rate (1) or sys rate (-1) is higher

    event_phase_ratio_df[f'{abbrev}_diff_ratio2dia'] = ((event_phase_ratio_df[dia_bin] / event_phase_ratio_df[f'{abbrev}_tot_ntrials']) - 
                                                        event_phase_ratio_df[f'{abbrev}_dia_prop2rr']) # event rate dia - proportion dia 
    event_phase_ratio_df[f'{abbrev}_check_ratio2dia'] = np.sign(event_phase_ratio_df[f'{abbrev}_diff_ratio2dia']) # check whether event rate (1) or dia rate (-1) is higher

    # Define conditions for coloring
    conditions = [
        (event_phase_ratio_df[f'{abbrev}_check_ratio2dia'] > 0) & (event_phase_ratio_df[f'{abbrev}_check_ratio2sys'] < 0),  # Q1: left up, overprop dia ratio
        (event_phase_ratio_df[f'{abbrev}_check_ratio2dia'] > 0) & (event_phase_ratio_df[f'{abbrev}_check_ratio2sys'] > 0),  # Q2: right up, non-defined
        (event_phase_ratio_df[f'{abbrev}_check_ratio2dia'] < 0) & (event_phase_ratio_df[f'{abbrev}_check_ratio2sys'] > 0)   # Q3: right down, overprop sys ratio
    ]                                                                                                                       # Q4: left down, non-defined (same as Q2)

    values = [1, 2, 3]  # Define corresponding values for coloring Q1, Q2/Q4, Q3
    event_phase_ratio_df['color'] = np.select(conditions, values, default=2)  # default color is 2 (non-defined)
    event_phase_ratio_df['color'] = event_phase_ratio_df['color'].astype('category') # convert to categorical

    # Define color palette (blue: dia, orange: sys, grey: non-defined)
    color_palette = {1: '#7AC5CD', 2: '#969696', 3: '#ff7f00'}

    # Create scatter plot
    plt.figure(figsize=(5, 5), dpi=300)
    ax = sns.scatterplot(data=event_phase_ratio_df,
                        x=f'{abbrev}_sys_ratio', y=f'{abbrev}_dia_ratio', # x-axis systolic ratio, y-axis diastolic ratio
                        hue='color', style='color', palette=color_palette,
                        markers=["o","^","s"], legend=False, s=50)

    # Dashed reference lines at x=1 and y=1 (expected normal distribution of phase-specific ratio)
    plt.axhline(1, linestyle="dashed", color="black", linewidth=0.8)
    plt.axvline(1, linestyle="dashed", color="black", linewidth=0.8)
    
    # Plot mean point of sys/dia ratio with error bars
    plt.errorbar(x=event_phase_ratio_agg["mean"][1], # sys ratio mean
                y=event_phase_ratio_agg["mean"][0], # dia ratio mean
                xerr=event_phase_ratio_agg["se"][1],
                yerr=event_phase_ratio_agg["se"][0],
                fmt="o", color="black", markersize=6, capsize=4)

    plt.xlim(xy_lim[0], xy_lim[1])
    plt.ylim(xy_lim[0], xy_lim[1])
    plt.xlabel("Systolic ratio", fontsize='large')
    plt.ylabel("Diastolic ratio", fontsize='large')

    sns.despine()

    #Save plot as SVG
    fig = os.path.join(plotting_dir, plot_fname)
    plt.savefig(fig, format="svg", bbox_inches='tight')

    plt.show()

    return ax, event_phase_ratio_df

In [ ]:
### FIGURE 3b ###
# Plot systolic vs. diastolic ratio of action onsets across all conditions
fig, event_phase_ratio_df = plot_sysdia_ratio(act_sysdia_ratio, 'act', 'Figure3b_sysdia_ratio_action.svg', xy_lim=[0.7,1.4])

In [ ]:
# Plot systolic vs. diastolic ratio of tone onsets across all conditions
fig2, tone_phase_ratio_df = plot_sysdia_ratio(tone_sysdia_ratio, 'tone', 'SupplFigure_sysdia_ratio_tone.svg', xy_lim=[0.7,1.4])

### **6d. Condition-wise analysis of systolic vs. diastolic ratio**

In [29]:
### Calculate normalized systolic & diastolic ratios for action onset - BasA condition ###

# Sub-select only BasA condition
beh_ecg_action_BasA = beh_ecg_action_clean[beh_ecg_action_clean[column_map['condition']].isin(['BasA'])]

# Compute the normalized ratios for the BasA condition only
act_sysdia_ratio_BasA = analyse_sysdia_ratio(beh_ecg_df=beh_ecg_action_BasA, abbrev=abbrev)

# Group-level binary analysis of action onset: two-sided paired t-test of sys vs. dia ratio
sysdia_ratio_ttest_BasA = pg.ttest(act_sysdia_ratio_BasA[f'{abbrev}_sys_ratio'], act_sysdia_ratio_BasA[f'{abbrev}_dia_ratio'], 
                                   paired=True, alternative='two-sided')
print("t-test on systolic-diastolic ratios of action onset in BasA only:\n\n", sysdia_ratio_ttest_BasA)

# Plot systolic vs. diastolic ratio of action onsets of BasA condition only
#fig_BasA, event_phase_ratio_df_BasA = plot_sysdia_ratio(act_sysdia_ratio_BasA, 'act', 'Figure4_sysdia_ratio_action_BasA.svg', xy_lim=[0.7,1.4])

t-test on systolic-diastolic ratios of action onset in BasA only:

                T  dof alternative    p_val           CI95   cohen_d     power  \
T_test  1.842123   43   two-sided  0.07236  [-0.01, 0.16]  0.508867  0.909745   

         BF10  
T_test  0.767  


In [30]:
### Calculate normalized systolic & diastolic ratios for action onset - OpA condition ###

# Sub-select only OpA condition
beh_ecg_action_OpA = beh_ecg_action_clean[beh_ecg_action_clean[column_map['condition']].isin(['OpA'])]

# Compute the normalized ratios for the OpA condition only
act_sysdia_ratio_OpA = analyse_sysdia_ratio(beh_ecg_df=beh_ecg_action_OpA, abbrev=abbrev)

# Group-level binary analysis of action onset: two-sided paired t-test of sys vs. dia ratio
sysdia_ratio_ttest_OpA = pg.ttest(act_sysdia_ratio_OpA[f'{abbrev}_sys_ratio'], act_sysdia_ratio_OpA[f'{abbrev}_dia_ratio'], 
                                   paired=True, alternative='two-sided')
print("t-test on systolic-diastolic ratios of action onset in OpA only:\n\n", sysdia_ratio_ttest_OpA)

# Plot systolic vs. diastolic ratio of action onsets of OpA condition only
#fig_OpA, event_phase_ratio_df_OpA = plot_sysdia_ratio(act_sysdia_ratio_OpA, 'act', 'Figure4_sysdia_ratio_action_OpA.svg', xy_lim=[0.7,1.4])

t-test on systolic-diastolic ratios of action onset in OpA only:

                T  dof alternative     p_val           CI95   cohen_d  \
T_test  1.708032   43   two-sided  0.094841  [-0.01, 0.16]  0.444606   

           power   BF10  
T_test  0.821936  0.622  


In [31]:
### Calculate normalized systolic & diastolic ratios for action onset - OpT condition ###

# Sub-select only OpT condition
beh_ecg_action_OpT = beh_ecg_action_clean[beh_ecg_action_clean[column_map['condition']].isin(['OpT'])]

# Compute the normalized ratios for the OpT condition only
act_sysdia_ratio_OpT = analyse_sysdia_ratio(beh_ecg_df=beh_ecg_action_OpT, abbrev=abbrev)

# Group-level binary analysis of action onset: two-sided paired t-test of sys vs. dia ratio
sysdia_ratio_ttest_OpT = pg.ttest(act_sysdia_ratio_OpT[f'{abbrev}_sys_ratio'], act_sysdia_ratio_OpT[f'{abbrev}_dia_ratio'], 
                                   paired=True, alternative='two-sided')
print("t-test on systolic-diastolic ratios of action onset in OpT only:\n\n", sysdia_ratio_ttest_OpT)

# Plot systolic vs. diastolic ratio of action onsets of OpT condition only
#fig_OpT, event_phase_ratio_df_OpT = plot_sysdia_ratio(act_sysdia_ratio_OpT, 'act', 'Figure4_sysdia_ratio_action_OpT.svg', xy_lim=[0.7,1.4])

t-test on systolic-diastolic ratios of action onset in OpT only:

                T  dof alternative     p_val           CI95   cohen_d  \
T_test  1.302067   43   two-sided  0.199825  [-0.03, 0.13]  0.338394   

           power   BF10  
T_test  0.592668  0.359  


In [32]:
### Calculate normalized systolic & diastolic ratios for tone onset - BasT condition ###

# Sub-select only BasT conditions
beh_ecg_tone_BasT = beh_ecg_tone_clean[beh_ecg_tone_clean[column_map['condition']].isin(['BasT'])]

# Compute the normalized ratios for the BasT condition only
tone_sysdia_ratio_BasT = analyse_sysdia_ratio(beh_ecg_df=beh_ecg_tone_BasT, abbrev=abbrev_tone)

# Group-level binary analysis of tone onset: two-sided paired t-test of sys vs. dia ratio
sysdia_ratio_tone_ttest_BasT = pg.ttest(tone_sysdia_ratio_BasT[f'{abbrev_tone}_sys_ratio'], tone_sysdia_ratio_BasT[f'{abbrev_tone}_dia_ratio'], 
                                   paired=True, alternative='two-sided')
print("t-test on systolic-diastolic ratios of tone onset in BasT only:\n\n", sysdia_ratio_tone_ttest_BasT)

# Plot systolic vs. diastolic ratio of tone onsets of BasT condition only
#fig_tone_BasT, event_phase_ratio_df_BasT = plot_sysdia_ratio(tone_sysdia_ratio_BasT, 'tone', 'Figure4_sysdia_ratio_tone_BasT.svg', xy_lim=[0.7,1.4])

t-test on systolic-diastolic ratios of tone onset in BasT only:

                T  dof alternative     p_val           CI95  cohen_d     power  \
T_test  1.113492   43   two-sided  0.271683  [-0.04, 0.14]  0.29637  0.484878   

         BF10  
T_test  0.291  


In [33]:
### Calculate normalized systolic & diastolic ratios for tone onset - OpT condition ###

# Sub-select only OpT conditions
beh_ecg_tone_OpT = beh_ecg_tone_clean[beh_ecg_tone_clean[column_map['condition']].isin(['OpT'])]

# Compute the normalized ratios for the OpT condition only
tone_sysdia_ratio_OpT = analyse_sysdia_ratio(beh_ecg_df=beh_ecg_tone_OpT, abbrev=abbrev_tone)

# Group-level binary analysis of tone onset: two-sided paired t-test of sys vs. dia ratio
sysdia_ratio_tone_ttest_OpT = pg.ttest(tone_sysdia_ratio_OpT[f'{abbrev_tone}_sys_ratio'], tone_sysdia_ratio_OpT[f'{abbrev_tone}_dia_ratio'], 
                                   paired=True, alternative='two-sided')
print("t-test on systolic-diastolic ratios of tone onset in OpT only:\n\n", sysdia_ratio_tone_ttest_OpT)

# Plot systolic vs. diastolic ratio of tone onsets of OpT condition only
#fig_tone_OpT, event_phase_ratio_df_OpT = plot_sysdia_ratio(tone_sysdia_ratio_OpT, 'tone', 'Figure4_sysdia_ratio_tone_OpT.svg', xy_lim=[0.6,1.4])

t-test on systolic-diastolic ratios of tone onset in OpT only:

                T  dof alternative     p_val         CI95   cohen_d     power  \
T_test  2.070898   43   two-sided  0.044405  [0.0, 0.16]  0.531939  0.931714   

         BF10  
T_test  1.133  


In [34]:
### Calculate normalized systolic & diastolic ratios for tone onset - OpA condition ###

# Sub-select only OpA conditions
beh_ecg_tone_OpA = beh_ecg_tone_clean[beh_ecg_tone_clean[column_map['condition']].isin(['OpA'])]

# Compute the normalized ratios for the OpA condition only
tone_sysdia_ratio_OpA = analyse_sysdia_ratio(beh_ecg_df=beh_ecg_tone_OpA, abbrev=abbrev_tone)

# Group-level binary analysis of tone onset: two-sided paired t-test of sys vs. dia ratio
sysdia_ratio_tone_ttest_OpA = pg.ttest(tone_sysdia_ratio_OpA[f'{abbrev_tone}_sys_ratio'], tone_sysdia_ratio_OpA[f'{abbrev_tone}_dia_ratio'], 
                                   paired=True, alternative='two-sided')
print("t-test on systolic-diastolic ratios of tone onset in OpA only:\n\n", sysdia_ratio_tone_ttest_OpA)

# Plot systolic vs. diastolic ratio of tone onsets of OpA condition only
#fig_tone_OpA, event_phase_ratio_df_OpA = plot_sysdia_ratio(tone_sysdia_ratio_OpA, 'tone', 'Figure4_sysdia_ratio_tone_OpA.svg', xy_lim=[0.6,1.4])

t-test on systolic-diastolic ratios of tone onset in OpA only:

                T  dof alternative     p_val           CI95  cohen_d     power  \
T_test  0.780295   43   two-sided  0.439492  [-0.05, 0.12]  0.20565  0.266079   

         BF10  
T_test  0.217  


## **7. Binary analysis: cardiac phase binning of intentional binding [Results 2.2.3 & 2.5.3]**

This section performs binary analysis of a given dependent variable (e.g., action binding, tone binding) acccording to the cardiac phase of the task event onset, i.e., binning into systole and diastole. In detail, the following two steps are included:

- **7a. Analyse intentional binding by binning task event onset in systole or diastole**: computes cardiac phase differences in the dependent variable (i.e., action binding or tone binding, respectively) by binning the task events of relevance (i.e., action or tone) into systole/diastole. This corresponds to Results section 2.2.3, as well as Figures 3c, 3d and 3e.
- **7b. Correlation of intentional binding with individual systolic/diastolic ratios**: performs correlation analysis between the individual normalized systolic and diastolic ratios for action/tone trials and the corresponding intentional binding measure (i.e., action binding and tone binding, respectively). This corresponds to Results section 2.5.3, as well as Figures 7a & 7b. 

### **7a. Analyse intentional binding by binning task event onset in systole or diastole**

After binning the task events of relevance (i.e., action or tone) into systole/diastole, cardiac phase differences in the dependent variable (i.e., action binding or tone binding, respectively) are computed. This is reported in Results section 2.2.3 with corresponding Figures 3c, 3d and 3e. 

In [35]:
############## 7a. Analyse intentional binding by binning task event onset in sys/dia ##############

# Define function to analyze differences in a dependent variable (e.g., action binding) according to 
# whether a task event onset was binned in systole or diastole
def analyze_sysdia_binding(beh_ecg_df, conditions, abbrev, depend_var, toff_method='neurokit', 
                           participant_col=column_map['participant'], condition_col=column_map['condition']):
    """
    Analyze the dependent variable from the intentional binding task (action or tone binding),
    according to the task event (i.e., action or tone onset) falling in systole and diastole phases.
    
    Parameters
    -----------
        beh_ecg_df : DataFrame
            DataFrame generated in the previous steps using the extract_ecg_features_event() function, containing
            preprocessed behavioral data and relevant ECG features around user-defined task-event. 
        conditions : tuple
            Tuple of two condition names to analyze, where first should be baseline condition and 
            second should be operant condition, e.g. ('BasA', 'OpA'). 
        abbrev : str
            Abbreviated name for the event type (e.g., 'act'), used as suffix for all the derived columns. 
            Should be the same as the one used in the extract_ecg_features_event() function to generate beh_ecg_df.  
        depend_var : str
            Column name including the dependent variable to be analyzed (e.g., 'JE_act')
        participant_col : str, default (column_map['participant'])
            Column name storing the participant IDs.   
        condition_col : str, default (column_map['condition'])
            Column name storing the condition names. 
        toff_method : str, default 'neurokit'
            Method to use for T-wave offset delineation and sys/dia segmentation, either 'neurokit' (default, 
            refers to NeuroKit2 'DWT' method) or 'tra' (Trapezium Area approach, see S4). 
    
    Returns
    -----------
    Tuple: (df_means, df_means_long, ttest_res)
        - df_means : Dataframe with means of dependent variable in systole vs. diastole (wide format)
        - df_means_long : Long-format dataframe for plotting
        - ttest_res : Paired samples t-test results comparing sys vs. dia

    """

    # Filter data according to sys/dia bins where task event falls
    if toff_method.lower() in ['neurokit', 'neurokit2', 'nk', 'dwt']:
        systole_df = beh_ecg_df[beh_ecg_df[f'{abbrev}_sys'] == 1]
        diastole_df = beh_ecg_df[beh_ecg_df[f'{abbrev}_dia'] == 1]
    elif toff_method.lower() in ['tra', 'trapezium']:
        systole_df = beh_ecg_df[beh_ecg_df[f'{abbrev}_sys_TRA'] == 1]
        diastole_df = beh_ecg_df[beh_ecg_df[f'{abbrev}_dia_TRA'] == 1]
    else:
        raise ValueError("Wrong method! T-offset delineation method should be either 'NeuroKit' or 'TRA'.")

    # Define function to calculate means of dependent variable for given conditions in systole vs. diastole
    def calculate_means(filtered_data, phase_name):
        filtered_data = filtered_data[filtered_data['condition'].isin(conditions)]  # Filter only selected conditions
        means = (
            filtered_data.groupby([participant_col, condition_col])[f'{depend_var}']
            .mean().unstack()
            .rename(columns={conditions[0]: f'JE_{conditions[0]}_{phase_name}', 
                             conditions[1]: f'JE_{conditions[1]}_{phase_name}'})
        )
        return means
    
    # Compute means for systole and diastole
    means_sys = calculate_means(systole_df, 'sys')
    means_dia = calculate_means(diastole_df, 'dia')

    # Merge the results and calculate action binding
    df_means = means_sys.join(means_dia, how='outer', lsuffix='_sys', rsuffix='_dia')
    df_means[f'{abbrev}_binding_sys'] = df_means[f'JE_{conditions[1]}_sys'] - df_means[f'JE_{conditions[0]}_sys']
    df_means[f'{abbrev}_binding_dia'] = df_means[f'JE_{conditions[1]}_dia'] - df_means[f'JE_{conditions[0]}_dia']

    # Convert to long format for plotting
    df_means_long = df_means[[f'{abbrev}_binding_sys', f'{abbrev}_binding_dia']].reset_index().melt(
        id_vars=participant_col, var_name='phase', value_name=f'{abbrev}_binding'
    )

    # Paired t-test   
    ttest_res = pg.ttest(df_means[f'{abbrev}_binding_sys'], df_means[f'{abbrev}_binding_dia'], 
                         paired=True, alternative='two-sided')
                                    

    return df_means, df_means_long, ttest_res

In [36]:
# Analyze action binding according to action onset in sys vs. dia
actbind_means, actbind_means_long, actbind_ttest = analyze_sysdia_binding(beh_ecg_df=beh_ecg_action_clean, conditions=('BasA', 'OpA'), 
                                                                          abbrev=abbrev, depend_var='JE_act')

# Summary of group-level results: action binding according to action onset in sys vs. dia
md("Binary analysis did not show a significant difference (t({})={}, p={}, Cohen's D={}) in action binding when actions were initiated at systole (M={}, SD={}) compared to diastole (M={}, SD={}).".format(
    actbind_ttest['dof'].iloc[0], round(actbind_ttest['T'].iloc[0],3), 
    round(actbind_ttest['p_val'].iloc[0],4), round(actbind_ttest['cohen_d'].iloc[0],3),
    round(actbind_means[f'act_binding_sys'].mean(),2), round(actbind_means[f'act_binding_sys'].std(),2), 
    round(actbind_means[f'act_binding_dia'].mean(),2), round(actbind_means[f'act_binding_dia'].std(),2)
))

Binary analysis did not show a significant difference (t(43)=1.618, p=0.1129, Cohen's D=0.258) in action binding when actions were initiated at systole (M=41.43, SD=48.16) compared to diastole (M=30.87, SD=32.33).

In [37]:
# Summary table of cardiac effects on JE in BasA/OpA and action binding

# Define comparisons for t-tests
comparisons_act = {"Judgement Error (ms) in BasA":  ('JE_BasA_sys', 'JE_BasA_dia'),          # Judgment Error in BasA condition
                   "Judgement Error (ms) in OpA":   ('JE_OpA_sys', 'JE_OpA_dia'),            # Judgment Error in OpA condition 
                   "Action Binding (ms)":           ('act_binding_sys', 'act_binding_dia')}  # Action binding (JE_OpA - JE_BasA)

summary_rows = []
for label, (sys_col, dia_col) in comparisons_act.items():
    sys_vals = actbind_means[sys_col]       # extract values for action onset in systole
    dia_vals = actbind_means[dia_col]       # extract values for action onset in diastole

    ttest_res = pg.ttest(sys_vals, dia_vals, paired=True, alternative='two-sided')  # paired t-test systole vs. diastole
    t_stat = ttest_res['T'].iloc[0]         
    p_val = ttest_res['p_val'].iloc[0]
    
    summary_rows.append({
        "Measure": label,
        "Systole mean (SD)": f"{sys_vals.mean():.2f} ({sys_vals.std():.2f})",
        "Diastole mean (SD)": f"{dia_vals.mean():.2f} ({dia_vals.std():.2f})",
        "T-statistic": f"{t_stat:.2f}",
        "p-value": f"{p_val:.4f}"
    })

    if label == 'Action Binding (ms)':
        actbind_cardeffect = sys_vals.mean() - dia_vals.mean()  # cardiac effect according to Koreki et al. (2022)

# Convert summary table to DataFrame and display
summary_table = pd.DataFrame(summary_rows)
print(summary_table.to_string(index=False))
print(f"\nCardiac effect (ms) on action binding: {round(actbind_cardeffect,3)}")

                     Measure Systole mean (SD) Diastole mean (SD) T-statistic p-value
Judgement Error (ms) in BasA     21.88 (68.99)      27.54 (63.17)       -1.18  0.2437
 Judgement Error (ms) in OpA     63.31 (62.57)      58.41 (64.05)        1.42  0.1638
         Action Binding (ms)     41.43 (48.16)      30.87 (32.33)        1.62  0.1129

Cardiac effect (ms) on action binding: 10.565


In [38]:
############## 7a. Analyse tone binding according to task event in sys/dia ##############

tonebind_means, tonebind_means_long, tonebind_ttest = analyze_sysdia_binding(beh_ecg_df=beh_ecg_tone_clean, conditions=('BasT', 'OpT'), 
                                                                             abbrev='tone', depend_var='JE_tone')

# Summary of group-level results: tone binding according to tone onset in sys vs. dia
md("Binary analysis did not show a significant difference (t({})={}, p={}, Cohen's D={}) in tone binding when tones occurred at systole (M={}, SD={}) compared to diastole (M={}, SD={}).".format(
    tonebind_ttest['dof'].iloc[0], round(tonebind_ttest['T'].iloc[0],3), 
    round(tonebind_ttest['p_val'].iloc[0],4), round(tonebind_ttest['cohen_d'].iloc[0],3),
    round(tonebind_means[f'tone_binding_sys'].mean(),2), round(tonebind_means[f'tone_binding_sys'].std(),2), 
    round(tonebind_means[f'tone_binding_dia'].mean(),2), round(tonebind_means[f'tone_binding_dia'].std(),2)
))

Binary analysis did not show a significant difference (t(43)=-0.448, p=0.6562, Cohen's D=0.04) in tone binding when tones occurred at systole (M=-87.32, SD=85.91) compared to diastole (M=-84.03, SD=76.89).

In [39]:
# Summary table of cardiac effects on JE in BasT/OpT and tone binding

# Define comparisons for t-tests
comparisons_tone = {"Judgement Error (ms) in BasT": ('JE_BasT_sys', 'JE_BasT_dia'),             # Judgment Error in BasT condition
                    "Judgement Error (ms) in OpT":  ('JE_OpT_sys', 'JE_OpT_dia'),               # Judgment Error in OpT condition 
                    "Tone Binding (ms)":            ('tone_binding_sys', 'tone_binding_dia')}   # Tone binding (JE_OpT - JE_BasT)

summary_rows = []
for label, (sys_col, dia_col) in comparisons_tone.items():
    sys_vals = tonebind_means[sys_col]       # extract values for action onset in systole
    dia_vals = tonebind_means[dia_col]       # extract values for action onset in diastole

    ttest_res = pg.ttest(sys_vals, dia_vals, paired=True, alternative='two-sided')  # paired t-test systole vs. diastole
    t_stat = ttest_res['T'].iloc[0]         
    p_val = ttest_res['p_val'].iloc[0]
    
    summary_rows.append({
        "Measure": label,
        "Systole mean (SD)": f"{sys_vals.mean():.2f} ({sys_vals.std():.2f})",
        "Diastole mean (SD)": f"{dia_vals.mean():.2f} ({dia_vals.std():.2f})",
        "T-statistic": f"{t_stat:.2f}",
        "p-value": f"{p_val:.4f}"
    })

    if label == 'Tone Binding (ms)':
        tonebind_cardeffect = sys_vals.mean() - dia_vals.mean()  # cardiac effect according to Koreki et al. (2022)

# Convert summary table to DataFrame and display
summary_table_tone = pd.DataFrame(summary_rows)
print(summary_table_tone.to_string(index=False))
print(f"\nCardiac effect (ms) on tone binding: {round(tonebind_cardeffect,3)}")

                     Measure Systole mean (SD) Diastole mean (SD) T-statistic p-value
Judgement Error (ms) in BasT     42.59 (64.81)      40.14 (64.59)        0.44  0.6654
 Judgement Error (ms) in OpT    -44.73 (97.49)     -43.89 (93.55)       -0.21  0.8350
           Tone Binding (ms)    -87.32 (85.91)     -84.03 (76.89)       -0.45  0.6562

Cardiac effect (ms) on tone binding: -3.292


In [ ]:
############## Summary arrow plot of action & tone binding across cardiac phases (Figure 3e) ##############

### FIGURE 3e ###

colpalette = ['#ff7f00', '#FFB870', '#7AC5CD', '#C4E3E9']

# Set figure size and dpi
fig, ax = plt.subplots(figsize=(7.5, 3), dpi=300)

# Draw a horizontal line and labels for time
time_axis = patches.FancyArrowPatch((-50, 0.8), (320, 0.8), color='k', arrowstyle='-|>', mutation_scale=8)
ax.add_patch(time_axis)
ax.text(x=125, y=0.83, s='250 ms', color='k', ha='center', va='center', fontsize=9) # delay label
ax.text(x=310, y=0.76, s='Time', color='k', ha='right', va='center', fontsize=9) # time axis label
int_arrow = patches.FancyArrowPatch((0, 0.77), (250, 0.77), color='grey', arrowstyle='<|-|>', mutation_scale=8, linewidth=0.8)
ax.add_patch(int_arrow)
ax.text(x=125, y=0.74, s='Actual interval', color='grey', ha='center', va='center', fontsize=7) # actual interval label

# Draw line for actual Action time (0 ms)
ax.axvline(x=0, color='k', linestyle='-', linewidth=4, ymin=0.75, ymax=0.85)
ax.text(x=0, y=0.92, s='Action', color='k', ha='center', va='center', fontsize=13)

# Draw line for actual Tone time (250ms)
ax.axvline(x=250, color='k', linestyle='-', linewidth=4, ymin=0.75, ymax=0.85)
ax.text(x=250, y=0.92, s='Tone', color='k', ha='center', va='center', fontsize=13)

# Draw perceived Baseline/Operant times for Systole 
ax.axvline(x=actbind_means['JE_BasA_sys'].mean(), color=colpalette[1], linestyle='-', linewidth=4, ymin=0.55, ymax=0.65)
ax.axvline(x=(250 + tonebind_means['JE_BasT_sys'].mean()), color=colpalette[1], linestyle='-', linewidth=4, ymin=0.55, ymax=0.65)
ax.axvline(x=actbind_means['JE_OpA_sys'].mean(), color=colpalette[0], linestyle='-', linewidth=4, ymin=0.55, ymax=0.65)
ax.axvline(x=(250 + tonebind_means['JE_OpT_sys'].mean()), color=colpalette[0], linestyle='-', linewidth=4, ymin=0.55, ymax=0.65)
ax.text(x=-80, y=0.6, s='Systole', color='k', ha='left', va='center', fontsize=11)

# Draw perceived Baseline/Operant times for Diastole
ax.axvline(x=actbind_means['JE_BasA_dia'].mean(), color=colpalette[3], linestyle='-', linewidth=4, ymin=0.35, ymax=0.45)
ax.axvline(x=(250 + tonebind_means['JE_BasT_dia'].mean()), color=colpalette[3], linestyle='-', linewidth=4, ymin=0.35, ymax=0.45)
ax.axvline(x=actbind_means['JE_OpA_dia'].mean(), color=colpalette[2], linestyle='-', linewidth=4, ymin=0.35, ymax=0.45)
ax.axvline(x=(250 + tonebind_means['JE_OpT_dia'].mean()), color=colpalette[2], linestyle='-', linewidth=4, ymin=0.35, ymax=0.45)
ax.text(x=-80, y=0.4, s='Diastole', color='k', ha='left', va='center', fontsize=11)

# Draw arrow and add text for Action Binding across Systole-Diastole
ax.arrow(x=(actbind_means['JE_BasA_sys'].mean()+1), y=0.6, dx=(actbind_means['JE_OpA_sys'].mean() - actbind_means['JE_BasA_sys'].mean())*0.89, dy=0, color=colpalette[0], 
         length_includes_head=True, head_width=0.015, head_length=2, linewidth=2.5)
ax.text(x=(actbind_means['JE_BasA_sys'].mean() + (actbind_means['JE_OpA_sys'].mean() - actbind_means['JE_BasA_sys'].mean())/2)*0.95, y=0.56, 
        s=f"{actbind_means['act_binding_sys'].mean():.2f} ({actbind_means['act_binding_sys'].std():.2f}) ms", color=colpalette[0], ha='center', va='center', fontsize=5)
ax.arrow(x=(actbind_means['JE_BasA_dia'].mean()+1), y=0.4, dx=(actbind_means['JE_OpA_dia'].mean() - actbind_means['JE_BasA_dia'].mean())*0.85, dy=0, color=colpalette[2], 
         length_includes_head=True, head_width=0.015, head_length=2, linewidth=2.5)
ax.text(x=(actbind_means['JE_BasA_dia'].mean() + (actbind_means['JE_OpA_dia'].mean() - actbind_means['JE_BasA_dia'].mean())/2)*0.84, y=0.36, 
        s=f"{actbind_means['act_binding_dia'].mean():.2f} ({actbind_means['act_binding_dia'].std():.2f}) ms", color=colpalette[2], ha='center', va='center', fontsize=5)

# Draw arrow and add text for Tone Binding across Systole-Diastole
ax.arrow(x=(250 + tonebind_means['JE_BasT_sys'].mean()), y=0.6, dx=(tonebind_means['JE_OpT_sys'].mean() - tonebind_means['JE_BasT_sys'].mean())*0.945, dy=0, color=colpalette[0], 
         length_includes_head=True, head_width=0.015, head_length=2, head_starts_at_zero=True, linewidth=2.5)
ax.text(x=((250 + tonebind_means['JE_BasT_sys'].mean())+((tonebind_means['JE_OpT_sys'].mean() - tonebind_means['JE_BasT_sys'].mean())/2)*0.95), y=0.56, 
         s=f"{tonebind_means['tone_binding_sys'].mean():.2f} ({tonebind_means['tone_binding_sys'].std():.2f}) ms", color=colpalette[0], ha='center', va='center', fontsize=5)
ax.arrow(x=(250 + tonebind_means['JE_BasT_dia'].mean()), y=0.4, dx=(tonebind_means['JE_OpT_dia'].mean() - tonebind_means['JE_BasT_dia'].mean())*0.94, dy=0, color=colpalette[2], 
         length_includes_head=True, head_width=0.015, head_length=2, head_starts_at_zero=True, linewidth=2.5)
ax.text(x=((250 + tonebind_means['JE_BasT_dia'].mean())+((tonebind_means['JE_OpT_dia'].mean() - tonebind_means['JE_BasT_dia'].mean())/2)*0.95), y=0.36,  
         s=f"{tonebind_means['tone_binding_dia'].mean():.2f} ({tonebind_means['tone_binding_dia'].std():.2f}) ms", color=colpalette[2], ha='center', va='center', fontsize=5)

# Set limits and remove axes
ax.set_xlim(-100, 350)
ax.set_ylim(0, 1)
ax.axis('off')

plt.tight_layout()

# Save plot 
fig5 = os.path.join(plotting_dir, "Figure3e_cardiac_arrow_summary.svg")
plt.savefig(fig5, format="svg")
plt.show()

In [41]:
##### Save the cardiac phase binning datasets for action binding & tone binding #####

# Save cardiac effect on action binding, averaged per participant in long format
actbind_dir = os.path.join(cardphase_dir, 'task-CardiAgIBTask_ecg_actbinding_avg.tsv')
actbind_means_long.to_csv(actbind_dir, index=False, sep='\t', na_rep="n/a")

# Save cardiac effect on tone binding, averaged per participant in long format
tonebind_dir = os.path.join(cardphase_dir, 'task-CardiAgIBTask_ecg_tonebinding_avg.tsv')
tonebind_means_long.to_csv(tonebind_dir, index=False, sep='\t', na_rep="n/a")

### **7b. Correlation of intentional binding with individual systolic/diastolic ratios**

This section performs correlation analysis between the individual normalized systolic and diastolic ratios for action/tone trials and the corresponding intentional binding measure (i.e., action binding and tone binding, respectively). This is reported in Results section 2.5.3 with the corresponding Figures 7a & 7b. 

In [42]:
############## 7b. Correlation of intentional binding with individual sys/dia ratios ##############

# Sub-select only BasA and OpA conditions
beh_ecg_action_BasAOpA = beh_ecg_action_clean[beh_ecg_action_clean[column_map['condition']].isin(['BasA', 'OpA'])]

# Run systolic/diastolic ratio analysis on BasA/OpA conditions
act_sysdia_ratio_BasAOpA = analyse_sysdia_ratio(beh_ecg_df=beh_ecg_action_BasAOpA, abbrev=abbrev)

# Append column with T-offset detection method used for each participant
method_map = beh_ecg_action_clean.groupby(column_map['participant'])['toff_method'].first()
act_sysdia_ratio_BasAOpA['toff_method'] = act_sysdia_ratio_BasAOpA[column_map['participant']].map(method_map)

# Summarize mJE (sd) for action and tone trials by condition and subject
IBmeasures_BasAOpA = beh_ecg_action_clean.groupby(["subjID", "condition"])['JE_act'].mean().unstack()
IBmeasures_BasAOpA = IBmeasures_BasAOpA.filter(items=['BasA', 'OpA']).rename(columns={'OpA': 'JE_OpA', 'BasA': 'JE_BasA'})
IBmeasures_BasAOpA['act_binding'] = IBmeasures_BasAOpA['JE_OpA'] - IBmeasures_BasAOpA['JE_BasA']

# Merge together systolic/diastolic ratios, and summary of JE_act per condition and subject
act_sysdia_ratio_actbind_BasAOpA = act_sysdia_ratio_BasAOpA.join(IBmeasures_BasAOpA, how='left', on='subjID')
act_sysdia_ratio_actbind_BasAOpA = act_sysdia_ratio_actbind_BasAOpA.join(actbind_means, how='left', on='subjID')

In [43]:
### Correlation of systolic ratio and action binding ###
sys_actbind_corr = pg.corr(x=act_sysdia_ratio_actbind_BasAOpA['act_sys_ratio'], y=act_sysdia_ratio_actbind_BasAOpA['act_binding'])

md("No significant correlation between individual systolic ratios and action binding (r({})={}, p={}).".format(
    (sys_actbind_corr['n'].iloc[0]-2), round(sys_actbind_corr['r'].iloc[0],3), round(sys_actbind_corr['p_val'].iloc[0],3)
))

No significant correlation between individual systolic ratios and action binding (r(42)=0.048, p=0.756).

In [44]:
### Correlation of diastolic ratio and action binding ###
dia_actbind_corr = pg.corr(x=act_sysdia_ratio_actbind_BasAOpA['act_dia_ratio'], y=act_sysdia_ratio_actbind_BasAOpA['act_binding'])

md("No significant correlation between individual diastolic ratios and action binding (r({})={}, p={}).".format(
    (dia_actbind_corr['n'].iloc[0]-2), round(dia_actbind_corr['r'].iloc[0],3), round(dia_actbind_corr['p_val'].iloc[0],3)
))

No significant correlation between individual diastolic ratios and action binding (r(42)=0.058, p=0.71).

In [ ]:
### FIGURE 7a ###
# Correlation plot of action binding and individual sys/dia ratios

plt.figure(figsize=(6, 6), dpi=300)

# Add horizontal line for group-level mean action binding effect
actbind_group_avg = 32.87
plt.axhline(actbind_group_avg, linestyle="dashed", color="k", linewidth=0.8, alpha=0.3)

# Scatter and regression line for act_sys_ratio
sns.regplot(data=act_sysdia_ratio_actbind_BasAOpA, x='act_sys_ratio', y='act_binding', marker= 's', 
            scatter_kws={'color': '#ff7f00', 'alpha': 0.7}, line_kws={'color': '#ff7f00'}, label="Systolic Ratio")

# Scatter and regression line for act_dia_ratio
sns.regplot(data=act_sysdia_ratio_actbind_BasAOpA, x='act_dia_ratio', y='act_binding',
            scatter_kws={'color': '#7AC5CD', 'alpha': 0.7}, line_kws={'color': '#7AC5CD'}, label="Diastolic Ratio")

# Labels and legend
plt.ylabel("Action Binding: OpA - BasA (ms)", fontsize='x-large')
plt.xlabel("Cardiac phase ratios", fontsize='x-large')
plt.legend()
sns.despine()

# Save plot 
fig10_1 = os.path.join(plotting_dir, "Figure7a_sysdia_ratio_actbind_correlation.svg")
plt.savefig(fig10_1, format="svg")
plt.show()

In [46]:
############## 7b. Correlation of dependent variable (i.e., tone binding) with individual sys/dia ratios ##############

# Sub-select only BasT and OpT conditions
beh_ecg_tone_BasTOpT = beh_ecg_tone_clean[beh_ecg_tone_clean[column_map['condition']].isin(['BasT', 'OpT'])]

# Run systolic/diastolic ratio analysis on BasT/OpT conditions
tone_sysdia_ratio_BasTOpT = analyse_sysdia_ratio(beh_ecg_df=beh_ecg_tone_BasTOpT, abbrev=abbrev_tone)

# Append the T-offset detection method used for each participant
method_map = beh_ecg_tone_clean.groupby(column_map['participant'])['toff_method'].first()
tone_sysdia_ratio_BasTOpT['toff_method'] = tone_sysdia_ratio_BasTOpT[column_map['participant']].map(method_map)


# Summarize mJE (sd) for action and tone trials by condition and subject
IBmeasures_BasTOpT = beh_ecg_tone_clean.groupby(["subjID", "condition"])['JE_tone'].mean().unstack()
IBmeasures_BasTOpT = IBmeasures_BasTOpT.filter(items=['BasT', 'OpT']).rename(columns={'OpT': 'JE_OpT', 'BasT': 'JE_BasT'})
IBmeasures_BasTOpT['tone_binding'] = IBmeasures_BasTOpT['JE_OpT'] - IBmeasures_BasTOpT['JE_BasT']

# Merge together systolic/diastolic ratios, and summary of JE_act per condition and subject
tone_sysdia_ratio_tonebind_BasTOpT = tone_sysdia_ratio_BasTOpT.join(IBmeasures_BasTOpT, how='left', on='subjID')
tone_sysdia_ratio_tonebind_BasTOpT = tone_sysdia_ratio_tonebind_BasTOpT.join(tonebind_means, how='left', on='subjID')

In [47]:
# Correlation of systolic ratio and tone binding
sys_tonebind_corr = pg.corr(x=tone_sysdia_ratio_tonebind_BasTOpT['tone_sys_ratio'], y=tone_sysdia_ratio_tonebind_BasTOpT['tone_binding'])

md("Strong correlation between individual systolic ratios and tone binding (r({})={}, p={}).".format(
    (sys_tonebind_corr['n'].iloc[0]-2), round(sys_tonebind_corr['r'].iloc[0],3), round(sys_tonebind_corr['p_val'].iloc[0],4)
))

Strong correlation between individual systolic ratios and tone binding (r(42)=0.52, p=0.0003).

In [48]:
# Correlation of diastolic ratio and tone binding
dia_tonebind_corr = pg.corr(x=tone_sysdia_ratio_tonebind_BasTOpT['tone_dia_ratio'], y=tone_sysdia_ratio_tonebind_BasTOpT['tone_binding'])

md("Significant correlation between individual diastolic ratios and tone binding (r({})={}, p={}).".format(
    (dia_tonebind_corr['n'].iloc[0]-2), round(dia_tonebind_corr['r'].iloc[0],3), round(dia_tonebind_corr['p_val'].iloc[0],4)
))

Significant correlation between individual diastolic ratios and tone binding (r(42)=-0.321, p=0.0338).

In [ ]:
### FIGURE 7b ###
# Correlation plot of tone binding and individual sys/dia ratios

plt.figure(figsize=(6, 6), dpi=300)

# Add horizontal line for group-level mean action binding effect
tonebind_group_avg = -88.18
plt.axhline(tonebind_group_avg, linestyle="dashed", color="k", linewidth=0.8, alpha=0.3)

# Scatter and regression line for tone_sys_ratio
sns.regplot(data=tone_sysdia_ratio_tonebind_BasTOpT, x='tone_sys_ratio', y='tone_binding', marker= 's', 
            scatter_kws={'color': '#ff7f00', 'alpha': 0.7}, line_kws={'color': '#ff7f00'}, label="Systolic Ratio")

# Scatter and regression line for tone_dia_ratio
sns.regplot(data=tone_sysdia_ratio_tonebind_BasTOpT, x='tone_dia_ratio', y='tone_binding',
            scatter_kws={'color': '#7AC5CD', 'alpha': 0.7}, line_kws={'color': '#7AC5CD'}, label="Diastolic Ratio")

# Labels and legend
plt.ylabel("Tone Binding: OpT - BasT (ms)", fontsize='x-large')
plt.xlabel("Cardiac phase ratios", fontsize='x-large')
plt.legend()
sns.despine()

# Save plot 
fig10_2 = os.path.join(plotting_dir, "Figure7b_sysdia_ratio_tonebind_correlation.svg")
plt.savefig(fig10_2, format="svg")
plt.show()

## **8. Circular analysis of task events across cardiac cycle [Results 2.2.1]**

This section performs circular analysis of task events around the cardiac cycle. This is reported in Results section 2.2.1, with corresponding Figure 3a and Extended Data Figure 1. In detail, the following steps are included:

- **8a. 1st level circular analysis (within-subject) - individual plots and Rayleigh test**: perform 1st-level circular analysis (within-subject) of task event onset across cardiac cycle. For each participant, the relative onset of each user-defined task event is computed within the cardiac cycle (i.e., the R-R interval) by assigning a radian value between 0 and 2 π (Ohl et al., 2016; Kunzendorf et al., 2019). Then, the participant-specific mean of the circular distribution of the task event onsets is tested against a uniform distribution using a Rayleigh test of uniformity. 
    - If `individual_plots` is set to `True`, individual circular plots with task event onsets and mean sys/dia duration within the R-R interval can be generated for each participant and saved in `derivatives/cardiac-phase-analysis/sub-XX/beh`.
- **8b. 2nd level circular analysis (between-subjects) - group-level plot and Rayleigh test**: perform 2nd-level circular analysis (between-subject) of task event onset across cardiac cycle. First, the group-level circular mean of task event onsets is derived by averaging the individual circular means, showing the average task event onset in the cardiac cycle across the group, and weighted by its length (mean resultant length ϱ) to reflect the spread of individual means around the circle. Then, the group-level mean resultant length ϱ is tested against the null hypothesis of uniform distribution using a Rayleigh test for uniformity (Pewsey et al., 2013). If ϱ gets sufficiently high to exceed a threshold value (i.e., the set of individual means is not spread evenly across the cardiac cycle), the data can be interpreted as too locally clustered to be consistent with a uniform distribution. 
    - If `group_plot` is set to `True`, group-level circular plot with circular mean and mean resultant length rho, as well as individual means of task event onsets and average sys/dia duration can be generated and saved in `results/datavisualization`.
- **8c. Non-parametric boostrapping analysis**: non-parametrically calculate 95% confidence intervals and significance through bootstrapping analysis (Ohl et al., 2016; Kunzendorf et al., 2019). Based on N of participants, a bootstrap sample of same size is drawn (with replacement). For each participant in the bootstrap sample, a circular density of task event onsets is computed and then the mean circular density of all participants in the bootstrap sample is derived. This bootstrap procedure is repeated `n_boot` times (recommended 10000 for full analysis). The 95% confidence intervals are derived as the 2.5% and 97.5% percentiles from the distribution of mean circular densities, so that significant deviations from a circular uniform are determined when the 95% confidence interval is outside the circular density of a uniform distribution. 
    - If `perform_bootstrap` is set to `True`, two serialized pickle files are saved in `derivatives/cardiac-phase-analysis`: `*_2ndlev_circular_bootstrap.pkl` containing results of each bootstrapping sample, and `*_2ndlev_circular_bootstrap_signif.pkl` containing CI, median, and significance markers 
- **8d. Group-level circular plot with bootstrapping CI and significance**: create the group-level circular plot with 2nd level circular mean and mean resultant length rho (central arrow, direction and length respectively), individual circular means of task event onsets (black dots), average circular distribution with 95% CI and significant segments, as determined by bootstrap analysis, colored differently based on average systolic (orange) or diastolic (blue) duration. This corresponds to **Figure 3a**.
- **8e. Condition-wise analysis of circular distributions of action onsets**: perform condition-wise comparisons between circular distribution of action onsets, divided by intentional binding condition. This corresponds to **Extended Data Figure 1**. 

In [50]:
############## General settings for circular analysis ##############

# Define function to draw segments in circular plot
def mean_circseg(rad):
    mean_rad = circular.mean_circular(rad, na_rm=True)[0]
    xcoord = math.sin(mean_rad)
    ycoord = math.cos(mean_rad)

    return xcoord, ycoord

# Define R-compatible parameters for circular plots
r_pi = robjects.r['pi'] # extract pi value
r_pi = r_pi[0]
rvector_xlim = robjects.FloatVector([-1.3, 1.3])
rvector_ylim = robjects.FloatVector([-1.3, 1.3])

### **8a. 1st level circular analysis (within-subject) - individual plots and Rayleigh test**

In [51]:
############## 8a. 1st level circular analysis (within-subject): individual plots and Rayleigh test ##############

# Define function to perform 1st level circular analysis of task event onset across cardiac cycle
def circular_analysis_1stlev(beh_ecg_df, abbrev, individual_plots=False, toff_method='neurokit',
                             participant_col=column_map['participant'], 
                             cardphase_dir=cardphase_dir, datatype_name=datatype_name):
    """
    Perform 1st-level circular analysis (within-subject) of task event onset across cardiac cycle. 

    For each participant, ECG-derived cardiac phase data is processed, including the relative onset 
    of the user-defined task event within the R-R interval and the mean systolic/diastolic onset 
    and offset timings (for plotting). Then, the participant-specific mean of the circular distribution 
    of the task event onsets is tested against a uniform distribution using a Rayleigh test of uniformity. 
    Optionally, individual circular plots with task event onsets and mean sys/dia duration within 
    the R-R interval can be generated in 'derivatives/cardiac-phase-analysis/sub-XX/beh'.

    Parameters
    ----------
    beh_ecg_df : DataFrame
        DataFrame containing preprocessed behavioral data and relevant ECG features.
    abbrev : str
        Abbreviated name for the event type (e.g., 'act'), used as a suffix for all derived columns.
    individual_plots : bool, default=False
        If True, generates individual circular plots of task event onsets within the cardiac cycle.
    toff_method : str, default='neurokit'
        Method for T-wave offset delineation ('neurokit' for NeuroKit2's DWT method, or 'tra' for 
        Trapezium Area approach).
    participant_col : str
        Column storing participant IDs.
    cardphase_dir : str
        Directory where the cardiac phase analysis results are stored.
    datatype_name : str
        BIDS-compatible datatype folder for individual plot sub-directory.

    Returns
    -------
    dict
        event_ecg_raytest : dict containing Rayleigh test results for each participant.
        
        Example structure:
        ```
        {
            participant_id: {
                'circular_mean': float,       # Circular mean (radians)
                'mean_length_rho': float,     # Mean resultant length
                'raytest_statistic': float,   # Rayleigh test statistic
                'raytest_pvalue': float       # p-value of Rayleigh test
            },
            ...
        }
        ```

    """

    # Initialize dict for storing Rayleigh test results for each subj
    event_ecg_raytest = {}

    # Loop over each participant
    for subj in beh_ecg_df[participant_col].unique():

        # Define directory for storing individual plots of 1st level circ analysis
        if individual_plots:
            indivplot_dir = os.path.join(cardphase_dir, f'sub-{subj}', datatype_name)
            if not os.path.exists(indivplot_dir):
                os.makedirs(indivplot_dir)

        # Filter data for the current participant
        subj_df = beh_ecg_df[beh_ecg_df[participant_col] == subj]
        ecg_radian_event = subj_df[f'event_ecg_radian_{abbrev}'].dropna() # extract position in rad of task event within RR interval

        # Convert from pandas to R dataframe
        ecg_radian_event_r = ro.conversion.py2rpy(ecg_radian_event)
  

        ############## 1st level circular analysis: individual plots ##############

        if individual_plots: 

            ##### Extract mean sys & dia onset-offset for current participant, relative to point zero at R-peak pre #####

            # Specify column names for T-wave offset and diastole onset according to selected method
            if toff_method.lower() in ['neurokit', 'neurokit2', 'nk', 'dwt']:
                toff = subj_df[f'Toffset_{abbrev}']
                dia_onset = subj_df[f'dia_onset_{abbrev}']
            elif toff_method.lower() in ['tra', 'trapezium']:
                toff = subj_df[f'Toffset_TRA_{abbrev}']
                dia_onset = subj_df[f'dia_onset_TRA_{abbrev}']
            else:
                raise ValueError("Wrong method! T-offset delineation method should be either 'NeuroKit' or 'TRA'.")

            # Systole
            sys_onset_s = subj_df[f'Soffset_{abbrev}'] - subj_df[f'Rpeak_pre_{abbrev}']         # sys onset at S-wave offset of R-peak pre (start of EP)
            sys_onset_rad = 2 * np.pi * subj_df[f'HR_1perSec_{abbrev}'] * sys_onset_s           # convert in rad
            sys_offset_s = toff - subj_df[f'Rpeak_pre_{abbrev}']                                # sys offset at T-wave offset (end of EP)
            sys_offset_rad = 2 * np.pi * subj_df[f'HR_1perSec_{abbrev}'] * sys_offset_s         # convert in rad

            # Diastole
            dia_onset_s = dia_onset - subj_df[f'Rpeak_pre_{abbrev}']                            # dia onset at +50ms after T-wave offset
            dia_onset_rad = 2 * np.pi * subj_df[f'HR_1perSec_{abbrev}'] * dia_onset_s           # convert in rad
            dia_offset_s = subj_df[f'Qonset_post_{abbrev}'] - subj_df[f'Rpeak_pre_{abbrev}']    # dia offset at Q-wave onset of R-peak post
            dia_offset_rad = 2 * np.pi * subj_df[f'HR_1perSec_{abbrev}'] * dia_offset_s         # convert in rad

            # Convert sys/dia onset and offset from pandas to R dataframe
            sys_onset_rad_r, sys_offset_rad_r, dia_onset_rad_r, dia_offset_rad_r = [
                ro.conversion.py2rpy(var) for var in [sys_onset_rad, sys_offset_rad, dia_onset_rad, dia_offset_rad]]

            # Define x and y coordinate of mean sys-dia onset and offset to draw segments in circular plot
            sys_onset_xcoord, sys_onset_ycoord = mean_circseg(sys_onset_rad_r)
            sys_offset_xcoord, sys_offset_ycoord = mean_circseg(sys_offset_rad_r)
            dia_onset_xcoord, dias_onset_ycoord = mean_circseg(dia_onset_rad_r)
            dia_offset_xcoord, dias_offset_ycoord = mean_circseg(dia_offset_rad_r)

            ##### Prepare individual circular plot #####

            # Calculate circular density
            res40 = circular.density_circular(ecg_radian_event_r, bw=40)

            # Define directory of plot
            plot_name = f"sub-{subj}_task-CardiAgIBTask_1stlev_circular_{abbrev}.png"
            plot_dir = os.path.normpath(os.path.join(indivplot_dir, plot_name))
            grdevices.png(file=plot_dir, width=2500, height=2500, res=300)

            # Circular plot with data points of task event onset for one participant
            circular.plot_circular(ecg_radian_event_r, units="radians", rotation="clock", zero=r_pi/2, stack=True, pch=16, 
                                   cex=1.5, ticks=True, tcl=0.08, col="black", xlim=rvector_xlim, ylim=rvector_ylim) 
            
            # Draw mean onset (solid) and offset (dashed) for systole (orange) and diastole (blue)
            graphics.segments(0,0,sys_onset_xcoord, sys_onset_ycoord, col="darkorange1", lty=1, lwd=4)       # segment for mean systole onset
            graphics.segments(0,0,sys_offset_xcoord, sys_offset_ycoord, col="darkorange1", lty=2, lwd=4)     # segment for mean systole offset
            graphics.segments(0,0,dia_onset_xcoord, dias_onset_ycoord, col="cadetblue3", lty=1, lwd=4)       # segment for mean diastole onset
            graphics.segments(0,0,dia_offset_xcoord, dias_offset_ycoord, col="cadetblue3", lty=2, lwd=4)     # segment for mean diastole offset
            
            # Draw arrow for circular mean (i.e., direction) and mean resultant length of task event onsets
            circular.arrows_circular(circular.mean_circular(ecg_radian_event_r), y=circular.rho_circular(ecg_radian_event_r), 
                                     lwd=5, zero=r_pi/2, rotation="clock", cex=1, col="black")       # arrow for circular mean and res. length
            
            # Add Kernel Density Estimate (KDE) of task event onset
            circular.lines_density_circular(res40, lwd=3, lty=2, col="darkgrey", rotation="clock", zero=r_pi/2) # add circular density 
            
            circular.axis_circular(units="radians", rotation="clock", zero=r_pi/2, template="none", cex=1.5)    # draw axes labels 
            grdevices.dev_off()


        ############## 1st level circular analysis: individual Rayleigh test ##############
    
        # Calculate circular mean and mean resultant length of task event onsets across cardiac cycle
        circ_mean_event = circular.mean_circular(ecg_radian_event_r)
        circ_rho_event = circular.rho_circular(ecg_radian_event_r)

        # Perform Rayleigh test of uniformity on task event onsets across cardiac cycle
        subj_raytesttmp = circular.rayleigh_test(ecg_radian_event_r)
        subj_raytest_pval = subj_raytesttmp[1][0]       # p-value of Rayleigh test
        subj_raytest_stat = subj_raytesttmp[0][0]       # mean resultant length (statistics)
        
        # Initialize the dict for the current subject
        if subj not in event_ecg_raytest:
            event_ecg_raytest[subj] = {}
        
        # Add the result to the dict
        event_ecg_raytest[subj] = {'circular_mean': circ_mean_event[0],
                                   'mean_length_rho': circ_rho_event[0],
                                   'raytest_statistic': subj_raytest_stat, 
                                   'raytest_pvalue': subj_raytest_pval}
        
    return event_ecg_raytest

In [52]:
# Perform 1st level (within-subject) circular analysis of action onset across cardiac cycle
act_ecg_raytest = circular_analysis_1stlev(beh_ecg_df=beh_ecg_action_clean, 
                                           abbrev=abbrev, individual_plots=False)

In [53]:
# Calculate FDR-corrected within-subject p-values of Rayleigh test
keys = list(act_ecg_raytest.keys())
pvals = np.array([act_ecg_raytest[subj]['raytest_pvalue'] for subj in keys])

rejected, pvals_fdr = fdrcorrection(pvals, alpha=0.05, method='indep') # apply FDR correction

for i, subj in enumerate(keys):
    act_ecg_raytest[subj]['raytest_pvalue_fdr'] = pvals_fdr[i]      # FDR-corrected p-values
    act_ecg_raytest[subj]['raytest_significant_fdr'] = rejected[i]  # significance after FDR correction (bool)


significant_count = sum(act_ecg_raytest[subj]['raytest_significant_fdr'] for subj in act_ecg_raytest)
significant_percent = significant_count / len(act_ecg_raytest) * 100

print(f'{significant_count} out of {len(act_ecg_raytest)} participants ({round(significant_percent,2)}%) show a significant FDR-corrected within-subject p-value for the Rayleigh test of action onsets across the cardiac cycle.')

0 out of 44 participants (0.0%) show a significant FDR-corrected within-subject p-value for the Rayleigh test of action onsets across the cardiac cycle.


### **8b. 2nd level circular analysis (between-subjects): group-level plot and Rayleigh test**

In [54]:
############## 8b. 2nd level circular analysis (between-subjects): group-level plot and Rayleigh test ##############

# Define function for performing 2nd level circular analysis and, optionally, create group-level plot
def circular_analysis_2ndlev(beh_ecg_df, abbrev, event_ecg_raytest, group_plot=True, 
                             circdens_bw=20, toff_method='neurokit', participant_col=column_map['participant'], 
                             plotting_dir=plotting_dir, save_svg=False, svg_fname=None):
    """
    Perform 2nd level circular analysis (between-subjects) of task event onset across cardiac cycle. 

    First, the group-level circular mean of task event onsets is derived by averaging the individual circular means, 
    and weighted by its mean resultant length rho to reflect the spread of individual means around the circle. 
    Then, the group-level mean resultant length rho is tested against the null hypothesis of uniform distribution 
    using a Rayleigh test for uniformity (Pewsey et al., 2013). Optionally, the group-level circular plot with individual 
    means of task event onsets, as well as group-level circular mean and mean resultant length and mean sys/dia duration 
    can be generated and saved in `results/datavisualization`.

    Parameters
    ----------
    beh_ecg_df : DataFrame
        DataFrame containing preprocessed behavioral data and relevant ECG features.
    abbrev : str
        Abbreviated name for the event type (e.g., 'act'), used as a suffix for all derived columns.
    event_ecg_raytest : dict
        Dictionary containing the 1st level (within-subject) circular analysis results, derived from
        the previous section circular_analysis_1stlev(). 
    group_plot : bool, default=True
        If True, generates the group-level circular plot of mean task event onsets within the cardiac cycle, 
        incl. arrow for circular mean and rho, mean sys/dia onsets and offsets and Kernel Density Estimate. 
    circdens_bw : int
        Smoothing bandwidth value for drawing Kernel Density Estimate (KDE) in plot; recommended to adjust
        according to own data, not too spiky nor too smooth. Default is 20. 
    toff_method : str, default='neurokit'
        Method for T-wave offset delineation ('neurokit' for NeuroKit2's DWT method, or 'tra' for 
        Trapezium Area approach).
    participant_col : str, default (column_map['participant'])
        Column name storing the participant IDs.   
    plotting_dir : str
        Directory where figures are stored.
    save_svg : bool, default=False
        If True, saves group-level plot as SVG suitable for publication.
    svg_fname : str
        If save_svg is set to True, defines the SVG file name.

    Returns
    -------
    Two dictionaries: 
        - sysdia_means_2ndlev: dict containing mean sys/dia onsets and offsets (in rad), both as pd.Series and R objects
        - raytest_2ndlev_res: dict containing 2nd level Rayleigh test results for the entire group, plus a list of all individual
        circular means, as well as group-level circular mean and rho values

    """ 

    ############## 2nd level circular analysis (between-subjects): plot ##############

    if group_plot: 

        ##### Calculate the group-level mean sys/dia onset and offset #####

        # Specify column names for T-wave offset and diastole onset according to selected method
        if toff_method.lower() in ['neurokit', 'neurokit2', 'nk', 'dwt']:
            toff = beh_ecg_df[f'Toffset_{abbrev}']
            dia_onset = beh_ecg_df[f'dia_onset_{abbrev}']
        elif toff_method.lower() in ['tra', 'trapezium']:
            toff = beh_ecg_df[f'Toffset_TRA_{abbrev}']
            dia_onset = beh_ecg_df[f'dia_onset_TRA_{abbrev}']
        else:
            raise ValueError("Wrong method! T-offset delineation method should be either 'NeuroKit' or 'TRA'.")
        
        # Systole
        sys_onset_s = beh_ecg_df[f'Soffset_{abbrev}'] - beh_ecg_df[f'Rpeak_pre_{abbrev}']       # sys onset at S-wave offset of R-peak pre (start of EP)
        sys_onset_rad = 2 * np.pi * beh_ecg_df[f'HR_1perSec_{abbrev}'] * sys_onset_s
        sys_offset_s = toff - beh_ecg_df[f'Rpeak_pre_{abbrev}']                                 # sys offset at T-wave offset (end of EP)
        sys_offset_rad = 2 * np.pi * beh_ecg_df[f'HR_1perSec_{abbrev}'] * sys_offset_s

        # Diastole
        dia_onset_s = dia_onset - beh_ecg_df[f'Rpeak_pre_{abbrev}']                            # dia onset at +50ms after T-wave offset
        dia_onset_rad = 2 * np.pi * beh_ecg_df[f'HR_1perSec_{abbrev}'] * dia_onset_s
        dia_offset_s = beh_ecg_df[f'Qonset_post_{abbrev}']  - beh_ecg_df[f'Rpeak_pre_{abbrev}'] # dia offset at Q-wave onset of R-peak post
        dia_offset_rad = 2 * np.pi * beh_ecg_df[f'HR_1perSec_{abbrev}'] * dia_offset_s

        # Convert sys/dia onset and offset from pandas to R dataframe
        sys_onset_rad_r, sys_offset_rad_r, dia_onset_rad_r, dia_offset_rad_r = [
            ro.conversion.py2rpy(var) for var in [sys_onset_rad, sys_offset_rad, dia_onset_rad, dia_offset_rad]]

        # Save list of circular means for each participant 
        circ_means_act = []

        for subj in beh_ecg_df[participant_col].unique():
            subj_circ_mean = event_ecg_raytest[subj]['circular_mean']
            circ_means_act.append(subj_circ_mean) # append value of circular mean for action onsets

        # Convert from pandas to R dataframe
        circ_means_act = pd.Series(circ_means_act)
        circ_means_act_r = ro.conversion.py2rpy(circ_means_act)

        # Define x and y coordinate of mean sys-dias onset and offset to draw segments in circular plot
        sys_onset_xcoord, sys_onset_ycoord = mean_circseg(sys_onset_rad_r)
        sys_offset_xcoord, sys_offset_ycoord = mean_circseg(sys_offset_rad_r)
        dia_onset_xcoord, dia_onset_ycoord = mean_circseg(dia_onset_rad_r)
        dia_offset_xcoord, dia_offset_ycoord = mean_circseg(dia_offset_rad_r)

        # Save group-level mean onset and offset of and diastole (in rad) in a dict
        sysdia_means_2ndlev = {'mean_syson': sys_onset_rad, 'mean_syson_r': sys_onset_rad_r,
                               'mean_sysoff': sys_offset_rad, 'mean_sysoff_r': sys_offset_rad_r,
                               'mean_diaon': dia_onset_rad, 'mean_diaon_r': dia_onset_rad_r,
                               'mean_diaoff': dia_offset_rad, 'mean_diaoff_r': dia_offset_rad_r}

        ##### Prepare individual circular plot #####
        
        # Calculate circular density
        res_2ndlev = circular.density_circular(circ_means_act_r, bw=int(circdens_bw))

        if save_svg:
            
            # Define directory of 2nd level circular plot (saved as SVG for publication)
            plot_dir_svg = os.path.join(plotting_dir, svg_fname)
            grdevices.svg(file=plot_dir_svg, width=8, height=8) 

        else:              

            # Define directory of 2nd level circular plot in `results/datavisualization` (saved as PNG and shown within notebook)
            plot_2ndlev_dir = os.path.join(plotting_dir, f"2ndlev_circular_{abbrev}.png")
            grdevices.png(file=plot_2ndlev_dir, width=5000, height=5000, res=600)

        # Circular plot with data points for all participants
        circular.plot_circular(circ_means_act_r, units="radians", rotation="clock", zero=r_pi/2, stack=False, pch=16, 
                               cex=1.5, ticks=True, tcl=0.08, col="black", axes=False, xlim=rvector_xlim, ylim=rvector_ylim)
        
        # Draw mean onsets (solid) and offsets (dashed) for systole (orange) and diastole (blue)
        graphics.segments(0,0,sys_onset_xcoord, sys_onset_ycoord, col="darkorange1", lty=1, lwd=4)      # segment for mean systole onset
        graphics.segments(0,0,sys_offset_xcoord, sys_offset_ycoord, col="darkorange1", lty=2, lwd=4)    # segment for mean systole offset
        graphics.segments(0,0,dia_onset_xcoord, dia_onset_ycoord, col="cadetblue3", lty=1, lwd=4)       # segment for mean diastole onset
        graphics.segments(0,0,dia_offset_xcoord, dia_offset_ycoord, col="cadetblue3", lty=2, lwd=4)     # segment for mean diastole offset
        
        # Draw arrow for circular mean (i.e., direction) and mean resultant length (i.e., arrow length) of task event onsets
        circular.arrows_circular(circular.mean_circular(circ_means_act_r), y=circular.rho_circular(circ_means_act_r), 
                                lwd=5, zero=r_pi/2, rotation="clock", cex=1, col="black")                           # arrow for circular mean and res. length
        
        # Add Kernel Density Estimate (KDE) of task event onset
        circular.lines_density_circular(res_2ndlev, lwd=3, lty=2, col="darkgrey", rotation="clock", zero=r_pi/2)    # add circular density 
        
        circular.axis_circular(units="radians", rotation="clock", zero=r_pi/2, template="none", cex=1.5)            # draw axes labels 
        grdevices.dev_off()

        if not save_svg: 
            # Load and display the 2nd level circular plot from PNG image
            plot = mpimg.imread(plot_2ndlev_dir)

            plt.figure(figsize=(8, 8))  
            plt.axis('off')  # Hide axes
            plt.imshow(plot)

    
    ############## 2nd level circular analysis (between-subjects): Rayleigh test ##############

    # Calculate circular mean and mean resultant length of task event onsets across cardiac cycle for entire group
    circ_mean_2lev = circular.mean_circular(circ_means_act_r)[0]
    circ_rho_2lev = circular.rho_circular(circ_means_act_r)[0]
    circ_sd_2lev = circular.sd_circular(circ_means_act_r)[0]

    # Perform Rayleigh test of uniformity on task event onsets across cardiac cycle for entire group
    raytesttmp_2lev = circular.rayleigh_test(circ_means_act_r)
    raytest_pval_2lev = raytesttmp_2lev[1][0]   # p-value of Rayleigh test
    raytest_stat_2lev = raytesttmp_2lev[0][0]   # mean resultant length (statistics)

    # Save 2nd level Rayleigh test results
    raytest_2ndlev_res = {'individual_circular_means': circ_means_act,  # list of individ. circular means for all participants
                          'circular_mean': circ_mean_2lev,              # group-level circular mean (direction)
                          'mean_length_rho': circ_rho_2lev,             # group-level mean resultant length (rho)
                          'circular_sd': circ_sd_2lev, 
                          'raytest_statistic': raytest_stat_2lev,
                          'raytest_pvalue': raytest_pval_2lev}
    
    print(f'Rayleigh Test of Uniformity: \n' + f'Test statistic: {raytest_stat_2lev}\n' +
          f'p-value: {raytest_pval_2lev}') 

    return sysdia_means_2ndlev, raytest_2ndlev_res

In [ ]:
# Perform 2nd level circular analysis of action onset across cardiac cycle
sysdia_means_2ndlev, raytest_2ndlev_res = circular_analysis_2ndlev(beh_ecg_df=beh_ecg_action_clean, abbrev=abbrev, 
                                                                   event_ecg_raytest=act_ecg_raytest, 
                                                                   group_plot=True, # save_svg=True, svg_fname=f"2ndlev_circular_{abbrev}.svg"
                                                                   circdens_bw=25)

### **8c. Non-parametric bootstrapping analysis**

In [56]:
############## 8c. Non-parametric bootstrapping analysis ##############

# Define function for bootstrapping analysis: non-parametical computation of confidence intervals 
# and significance for circular data (based on Ohl et al., 2016)
def bootstrap_circular(beh_ecg_df, abbrev, n_boot, bw_param=20, 
                       participant_col=column_map['participant'], cardphase_dir=cardphase_dir):
    """
    Perform a non-parametric bootstrap analysis on circular data for computing confidence intervals and significance.
    
    * From the original N of participants, a bootstrap sample of same size is drawn (with replacement)
    * For each participant in the bootstrap sample, a circular density (with specified bandwidth) of task event onsets is computed
    * The mean circular density across all participants in the bootstrap sample is computed
    * This bootstrap procedure is repeated n_boot times (recommended 10000 for full analysis)
    * 95% confidence intervals are computed as 2.5% and 97.5% percentiles from the distribution of mean circular densities. 
    * Significance is determined when the 95% CI is outside the circular density of a uniform distribution.

    (Stella's annotation: play around with bw. Make chi-square tests and check whether observed coupling is observed for all kinds of bw parameters)

    Parameters
    ----------
    beh_ecg_df : DataFrame
        DataFrame containing preprocessed behavioral data and relevant ECG features.
    abbrev : str
        Abbreviated name for the event type (e.g., 'act'), used as a suffix for all derived columns.
    n_boot : int
        Number of bootstrap samples to be computed (recommended 10000 for full analysis, 100 for testing).
    bw_param : int, default 20
        Bandwidth parameter for Kernel Density Estimation.
    participant_col : str, default (column_map['participant'])
        Column name storing the participant IDs.   
    cardphase_dir : str
        Directory where the cardiac phase analysis results are stored.
    
    Returns:
    ----------
       Two serialized .pkl files are saved in the specified cardphase_dir: 
        - '*_2ndlev_circular_bootstrap.pkl' containing results of each bootstrapping sample
        - '*_2ndlev_circular_bootstrap_signif.pkl' containing CI, median, and significance markers
    """
    
    # Extract unique participant list
    subj_list = beh_ecg_df[participant_col].unique()
    
    # Output storage
    out = []
    
    # Bootstrap procedure
    for i in tqdm(range(n_boot), desc="Bootstrapping"):
        
        # Draw bootstrap sample of same size as original sample, with replacement 
        subj_list_boot = np.random.choice(subj_list, size=len(subj_list), replace=True)
        buffer = [] # tmp buffer for subj data
        
        for subj in subj_list_boot:
            # Filter data for this participant
            data = beh_ecg_df.loc[beh_ecg_df[participant_col] == subj, 
                                  f'event_ecg_radian_{abbrev}'].values
            data = data[~np.isnan(data)]  # remove NaNs

            if len(data) > 0:
                # Convert participant data to R circular object
                r_data = robjects.FloatVector(data)
                r_circular_data = circular.circular(r_data, type="angles", units="radians", 
                                                    modulo="2pi", zero=r_pi/2, rotation="clock")

                # Compute circular density
                kde_res = robjects.r('density.circular')(r_circular_data, bw=bw_param)
                density_values = np.array(kde_res[2])  # Extract density values
                buffer.append(density_values)
        
        if buffer:
            # Compute mean circular density across participants
            buffer_mean = np.mean(np.array(buffer), axis=0)
            out.append(buffer_mean)
    
    out = np.array(out) # convert to np array

    # Compute 95% confidence intervals and median
    ci_lower = np.percentile(out, 2.5, axis=0)
    ci_upper = np.percentile(out, 97.5, axis=0)
    ci_median = np.percentile(out, 50, axis=0)

    # Determine significant segments where the CI are outside uniform distribution
    markup_upper = np.where(ci_lower > circular.dcircularuniform(1)) # where ci_lower (2.5%) outside uniform -> signif. more event onsets
    markup_lower = np.where(ci_upper < circular.dcircularuniform(1)) # where ci_upper (97.5%) outside uniform -> signif. less event onsets

    # Ensure directory exists
    os.makedirs(cardphase_dir, exist_ok=True)

    # Save array with results of each bootstrapping sample
    bootres_dir = os.path.join(cardphase_dir, f'task-CardiAgIBTask_2ndlev_circular_bootstrap_{abbrev}.pkl')
    with open(bootres_dir, "wb") as f:
        pickle.dump(out, f)

    # Save dict with CI, median, and significance markers
    bootres_signif_dir = os.path.join(cardphase_dir, f'task-CardiAgIBTask_2ndlev_circular_bootstrap_signif_{abbrev}.pkl')
    with open(bootres_signif_dir, "wb") as f:
        pickle.dump({"ci_lower": ci_lower, "ci_upper": ci_upper, "ci_median": ci_median,
                     "markup_upper": markup_upper, "markup_lower": markup_lower}, f)


    return print(f"Bootstrap complete. Results saved in {cardphase_dir} folder.")

In [57]:
# Perform the bootstrapping analysis for action onsets
perform_bootstrap = False  # set to True only when the bootstrap analysis is required (might take very long!)

if perform_bootstrap:
    bootstrap_circular(beh_ecg_df=beh_ecg_action_tra, 
                       abbrev=abbrev, 
                       n_boot=10000,        # should be set to 10000 iterations for actual analyses
                       bw_param=20)

### **8d. Group-level circular plot with bootstrapping CI and significance**

In [61]:
############## 8d. Group-level circular plot with bootstrapping CI and significance ##############

# Define function to plot 2nd level circular analysis results, including: group-level circular mean and 
# mean resultant length rho of task event onsets; average circular distribution with 95% CI and significant 
# segments, as determined by bootstrap analysis; individual points for mean task event onsets
def plot_bootstrap_circular_2ndlev(beh_ecg_df, abbrev, sysdia_means_2ndlev=sysdia_means_2ndlev, 
                                   participant_col=column_map['participant'], cardphase_dir=cardphase_dir):

    # Define color palette
    colors_map = {'sys': 'darkorange1', 'dia': 'cadetblue3', 'non-def': 'grey'}

    # Load info about bootstrapping analysis significance from .pkl file (CI, median, signif. segments)
    bootres_signif_dir = os.path.join(cardphase_dir, f'task-CardiAgIBTask_2ndlev_circular_bootstrap_signif_{abbrev}.pkl')
    if os.path.exists(bootres_signif_dir): # check if the file exists before opening
        bootres_dict = pd.read_pickle(bootres_signif_dir)  
    else:
        print(f"Warning: The file '{bootres_signif_dir}' does not exist.")
        bootres_dict = None  # Optional: Set to None or handle accordingly

    
    # Normalize CI and median and convert to R objects
    ci_median_r = robjects.FloatVector(bootres_dict['ci_median'] / circular.dcircularuniform(1))
    ci_lower_r = robjects.FloatVector(bootres_dict['ci_lower'] / circular.dcircularuniform(1))
    ci_upper_r = robjects.FloatVector(bootres_dict['ci_upper'] / circular.dcircularuniform(1))

    # Save list of circular means for each participant 
    circ_means = []

    for subj in beh_ecg_df[participant_col].unique():
        subj_circ_mean = act_ecg_raytest[subj]['circular_mean']
        circ_means.append(subj_circ_mean) # append value of circular mean for action onsets

    # Convert from pandas to R dataframe
    circ_means = pd.Series(circ_means)
    circ_means_r = ro.conversion.py2rpy(circ_means)

    # Calculate circular density of group-level task event onsets
    res_2ndlev = circular.density_circular(circ_means_r, bw=20)

    plot_dir = plotting_dir + f'\\Figure3a_bootstrap_circular_{abbrev}.png'
    grdevices.png(file=plot_dir, width=5000, height=5000, res=600)
    # plot_dir_svg = plotting_dir + f'\\Figure3a_bootstrap_circular_{abbrev}.svg'     # uncomment to save SVG
    # grdevices.svg(file=plot_dir_svg, width=8, height=8)                # uncomment to save SVG

    # Generate the sequence of x-values of circular distribution (in rad) and convert into R object
    seq_x = np.linspace(0, 2 * np.pi, num=len((bootres_dict['ci_median'])))
    seq_x_r = robjects.FloatVector(seq_x)

    circular.plot_circular(circ_means_r, units="radians", rotation="clock", zero=r_pi/2, stack=False, pch=16, 
                           cex=1.5, ticks=False, tcl=0.08, col="black", axes=False, xlim=rvector_xlim, ylim=rvector_ylim)
    circular.arrows_circular(circular.mean_circular(circ_means_r), y=circular.rho_circular(circ_means_r), 
                             lwd=5, zero=r_pi/2, rotation="clock", cex=1, col="black")                              # arrow for circular mean and res. length
    # circular.lines_density_circular(res_2ndlev, lwd=3, lty=2, col="darkgrey", rotation="clock", zero=r_pi/2)      # add circular density 
    circular.axis_circular(units="radians", rotation="clock", zero=r_pi/2, template="none", cex=1.5)                # draw axes labels 
    circular.lines_circular(seq_x_r, ci_lower_r, join=True, col="grey", lwd=1.5, rotation="clock", offset=0, zero=r_pi/2)
    circular.lines_circular(seq_x_r, ci_upper_r, join=True, col="grey", lwd=1.5, rotation="clock", offset=0, zero=r_pi/2)

    # Add mean circular density of bootstrapping sample
    circular.lines_circular(seq_x_r, ci_median_r, join=True, col="grey", lwd=4, rotation="clock", offset=0, zero=r_pi/2)

    # Add systolic (orange) and diastolic (blue) segments of mean circular density
    mean_syson = np.array(circular.mean_circular(sysdia_means_2ndlev['mean_syson_r'], na_rm=True))[0]
    mean_sysoff = np.array(circular.mean_circular(sysdia_means_2ndlev['mean_sysoff_r'], na_rm=True))[0]
    mean_diaon = np.array(circular.mean_circular(sysdia_means_2ndlev['mean_diaon_r'], na_rm=True))[0]
    mean_diaon = mean_diaon % (2 * np.pi)
    mean_diaoff = np.array(circular.mean_circular(sysdia_means_2ndlev['mean_diaoff_r'], na_rm=True))[0]
    mean_diaoff = mean_diaoff % (2 * np.pi)

    # Compute indices for systolic and diastolic segments
    msyspos = np.where((seq_x >= mean_syson) & (seq_x <= mean_sysoff))[0]
    mdiapos = np.where((seq_x >= mean_diaon) & (seq_x <= mean_diaoff))[0]

    sys_x = robjects.FloatVector(seq_x[msyspos])
    sys_y = robjects.FloatVector((bootres_dict['ci_median'] / circular.dcircularuniform(1))[msyspos])
    dia_x = robjects.FloatVector(seq_x[mdiapos])
    dia_y = robjects.FloatVector((bootres_dict['ci_median'] / circular.dcircularuniform(1))[mdiapos])

    # Plot systolic (orange) and diastolic (blue) segments
    circular.lines_circular(sys_x, sys_y, join=False, col=colors_map['sys'], lwd=4, rotation="clock", offset=0, zero=r_pi/2)
    circular.lines_circular(dia_x, dia_y, join=False, col=colors_map['dia'], lwd=4, rotation="clock", offset=0, zero=r_pi/2)

    # Extract significant density segments outside upper CI (where significantly more task event onsets)
    markup_upper_xcoord = robjects.FloatVector(seq_x[bootres_dict['markup_upper'][0]]) 
    markup_upper_ycoord = robjects.FloatVector((bootres_dict['ci_median'] / circular.dcircularuniform(1))[bootres_dict['markup_upper'][0]])

    # Compute differences between consecutive x-coordinates
    diff_x = np.diff(markup_upper_xcoord)

    # Define a threshold to detect discontinuities (e.g., a large jump in radians)
    threshold = np.pi / 2  # Adjust if needed

    # Find split points where the jump is too large
    split_indices = np.where(diff_x > threshold)[0] + 1  # +1 to shift to the next element

    # Split the x and y coordinates at these discontinuities
    x_segments = np.split(markup_upper_xcoord, split_indices)
    y_segments = np.split(markup_upper_ycoord, split_indices)

    # Convert systolic and diastolic indices to sets for easy checking
    systolic_indices = set(msyspos)
    diastolic_indices = set(mdiapos)

    # Convert each segment into an R FloatVector and plot separately
    for x_seg, y_seg in zip(x_segments, y_segments):
        x_r = robjects.FloatVector(x_seg)
        y_r = robjects.FloatVector(y_seg)

        # Determine if this segment is systolic or diastolic
        segment_indices = set(np.searchsorted(seq_x, x_seg))  # Find corresponding indices in seq_x

        if segment_indices.issubset(systolic_indices):  # Entire segment in systole
            color = colors_map['sys']
        elif segment_indices.issubset(diastolic_indices):  # Entire segment in diastole
            color = colors_map['dia']
        else:
            color = colors_map['non-def']  # If it spans both or is unclear

        # Plot the segment with the chosen color
        circular.lines_circular(x_r, y_r, join=False, col=color, lwd=8, rotation="clock", offset=0, zero=r_pi/2)

    grdevices.dev_off()

    # Load and display the 2nd level circular plot
    plot = mpimg.imread(plot_dir)

    plt.figure(figsize=(8, 8))  
    plt.axis('off')  # Hide axes
    plt.imshow(plot)

    print(f"Significant segments: {bootres_dict['markup_upper']}")

In [ ]:
### FIGURE 3a ###

# Plot 2nd level circular plot with bootstrap results
plot_bootstrap_circular_2ndlev(beh_ecg_df=beh_ecg_action_tra, abbrev=abbrev)

### **8e. Condition-wise analysis of circular distributions of action onsets**

In [62]:
##### 8e. Circular analysis of action onsets in BasA #####

beh_ecg_action_tra_BasA = beh_ecg_action_tra[beh_ecg_action_tra['condition'] == 'BasA']
act_ecg_raytest_BasA = circular_analysis_1stlev(beh_ecg_df=beh_ecg_action_tra_BasA, abbrev=abbrev, individual_plots=False, toff_method='TRA')

# Perform 2nd level circular analysis of action onset across cardiac cycle in BasA condition
sysdia_means_2ndlev_BasA, raytest_2ndlev_res_BasA = circular_analysis_2ndlev(beh_ecg_df=beh_ecg_action_tra_BasA, abbrev=abbrev, event_ecg_raytest=act_ecg_raytest_BasA, 
                                                                             group_plot=True, circdens_bw=25, toff_method='TRA', plotting_dir=plotting_dir, 
                                                                             save_svg=True, svg_fname='ExDFigure1a_circular_act_BasA.svg')

Rayleigh Test of Uniformity: 
Test statistic: 0.2035827283894142
p-value: 0.16174498573237547


In [63]:
##### 8e. Circular analysis of action onsets in OpA #####

beh_ecg_action_tra_OpA = beh_ecg_action_tra[beh_ecg_action_tra['condition'] == 'OpA']
act_ecg_raytest_OpA = circular_analysis_1stlev(beh_ecg_df=beh_ecg_action_tra_OpA, abbrev=abbrev, individual_plots=False, toff_method='TRA')

# Perform 2nd level circular analysis of action onset across cardiac cycle in OpA condition
sysdia_means_2ndlev_OpA, raytest_2ndlev_res_OpA = circular_analysis_2ndlev(beh_ecg_df=beh_ecg_action_tra_OpA, abbrev=abbrev, event_ecg_raytest=act_ecg_raytest_OpA, 
                                                                           group_plot=True, circdens_bw=25, toff_method='TRA', plotting_dir=plotting_dir, 
                                                                           save_svg=True, svg_fname='ExDFigure1b_circular_act_OpA.svg')

Rayleigh Test of Uniformity: 
Test statistic: 0.14233297253556895
p-value: 0.41241729497025


In [64]:
##### 8e. Circular analysis of action onsets in OpT #####

beh_ecg_action_tra_OpT = beh_ecg_action_tra[beh_ecg_action_tra['condition'] == 'OpT']
act_ecg_raytest_OpT = circular_analysis_1stlev(beh_ecg_df=beh_ecg_action_tra_OpT, abbrev=abbrev, individual_plots=False, toff_method='TRA')

# Perform 2nd level circular analysis of action onset across cardiac cycle in OpT condition
sysdia_means_2ndlev_OpT, raytest_2ndlev_res_OpT = circular_analysis_2ndlev(beh_ecg_df=beh_ecg_action_tra_OpT, abbrev=abbrev, event_ecg_raytest=act_ecg_raytest_OpT, 
                                                                           group_plot=True, circdens_bw=25, toff_method='TRA', plotting_dir=plotting_dir, 
                                                                           save_svg=True, svg_fname='ExDFigure1c_circular_act_OpT.svg')

Rayleigh Test of Uniformity: 
Test statistic: 0.14292599430084552
p-value: 0.4093628787487058


In [65]:
# Watson's two-samples tests for comparisons of action onset distribution across conditions

indiv_means_BasA = ro.conversion.py2rpy(raytest_2ndlev_res_BasA['individual_circular_means'])
indiv_means_OpA = ro.conversion.py2rpy(raytest_2ndlev_res_OpA['individual_circular_means'])
indiv_means_OpT = ro.conversion.py2rpy(raytest_2ndlev_res_OpT['individual_circular_means']) 

watson_test_BasAOpA = circular.watson_two_test(indiv_means_BasA, indiv_means_OpA, alpha=0.01)
watson_test_BasAOpT = circular.watson_two_test(indiv_means_BasA, indiv_means_OpT, alpha=0.01)
watson_test_OpAOpT = circular.watson_two_test(indiv_means_OpA, indiv_means_OpT, alpha=0.01)

watson_res_BasAOpA = str(watson_test_BasAOpA[3])
watson_res_BasAOpT = str(watson_test_BasAOpT[3])
watson_res_OpAOpT = str(watson_test_OpAOpT[3])

print(f"Watson's U2 test - BasA vs. OpA: {watson_res_BasAOpA}")
print(f"Watson's U2 test - BasA vs. OpT: {watson_res_BasAOpT}")
print(f"Watson's U2 test - OpA vs. OpT: {watson_res_OpAOpT}")

Watson's U2 test - BasA vs. OpA: 
      Watson's Two-Sample Test of Homogeneity 

Test Statistic: 0.031 
Level 0.01 Critical Value: 0.268 
Do Not Reject Null Hypothesis 


Watson's U2 test - BasA vs. OpT: 
      Watson's Two-Sample Test of Homogeneity 

Test Statistic: 0.0239 
Level 0.01 Critical Value: 0.268 
Do Not Reject Null Hypothesis 


Watson's U2 test - OpA vs. OpT: 
      Watson's Two-Sample Test of Homogeneity 

Test Statistic: 0.0445 
Level 0.01 Critical Value: 0.268 
Do Not Reject Null Hypothesis 




## **9. Pre- and post-event R-R interval changes [Results 2.5.1]**

This section analyzes changes in the duration of R-R intervals before and after the task event of interest, indicative of cardiac deceleration and/or acceleration. In detail: 

- **9a. Extract R-R intervals before and after action**: extract R-R intervals before (T-2, T-1) and after (T+1, T+2, T+3) the R-R interval where the task event (i.e., action) occurs (T0); in the CardiAg intentional binding task, the event of interest is the action onset, and perform a two-way repeated-measure ANOVA 6 (Timepoint) x 3 (Condition) on R-R interval duration. This is reported in Results section 2.5.1, with corresponding Figure 6a. 
- **9b. Pre-action cardiac deceleration**: perform slope analysis of cardiac deceleration (z-scored) preceding action onset (i.e., from T-1 to T0) between intentional binding conditions. This corresponds to Figure 6b.
- **9c. Post-action cardiac acceleration**: perform slope analysis of cardiac acceleration (z-scored) following action onset (i.e., from T0 to T+1) between intentional binding conditions. This corresponds to Figure 6c.

In [66]:
############## 9a. Extract R-R intervals before and after action ##############

# Define function for extracting R-R intervals before (T-2, T-1) and after (T+1, T+2, T+3) 
# the R-R interval where the task event occurs (T0)
def extract_prepost_rr(beh_ecg_df, abbrev, column_map=column_map): 
    
    beh_ecg_rr = [] # initialize empty list to store dfs for all participants

    # Extract relevant ECG features for the current participant
    for subj in beh_ecg_df[column_map['participant']].unique():
        subj_ecg_data = subj_qrst.get(f'sub-{subj}', {})
        rpeaks_s = subj_ecg_data.get('rpeaks_s', [])

        # Filter df with preprocessed beh data for current participant
        subj_rr_df = beh_ecg_df[beh_ecg_df[column_map['participant']] == subj].copy()

        # Specify relevant column names for ecg-beh analysis based on the event abbreviation
        columns = {name: f"{name}_{abbrev}" for name in [
            'Rpeak_Tminus2', 'Rpeak_Tminus1', 'Rpeak_Tplus1', 'Rpeak_Tplus2', 'Rpeak_Tplus3',
            'RR_s_Tminus2', 'RR_s_Tminus1', 'RR_s_Tplus1', 'RR_s_Tplus2', 'RR_s_Tplus3'
        ]}

        subj_rr_df = subj_rr_df.assign(**{col: np.nan for col in columns.values()})

        # Iterate through each row to extract/calculate various ECG features relative to the task event
        for index, row in subj_rr_df.iterrows():
            rpeak_pre = row[f'Rpeak_pre_{abbrev}']      # specify column with timestamps for the R-peak pre (just before task event onset)
            rpeak_post = row[f'Rpeak_post_{abbrev}']    # specify column with timestamps for the R-peak post (just after task event onset)
            
            # Find the two R-peaks before the R-R interval of task event onset (T-2, T-1)
            rpeak_Tminus1 = max([peak_m1 for peak_m1 in rpeaks_s if peak_m1 < rpeak_pre], default=np.nan)
            rpeak_Tminus2 = max([peak_m2 for peak_m2 in rpeaks_s if peak_m2 < rpeak_Tminus1], default=np.nan)
            subj_rr_df.at[index, columns['Rpeak_Tminus1']] = rpeak_Tminus1       # R-peak at T-1 before R-R interval of task event onset
            subj_rr_df.at[index, columns['Rpeak_Tminus2']] = rpeak_Tminus2       # R-peak at T-2 before R-R interval of task event onset

            # Find the three R-peaks after the R-R interval of task event onset (T+1, T+2, T+3)
            rpeak_Tplus1 = min([peak_p1 for peak_p1 in rpeaks_s if peak_p1 > rpeak_post], default=np.nan)
            rpeak_Tplus2 = min([peak_p2 for peak_p2 in rpeaks_s if peak_p2 > rpeak_Tplus1], default=np.nan)
            rpeak_Tplus3 = min([peak_p3 for peak_p3 in rpeaks_s if peak_p3 > rpeak_Tplus2], default=np.nan)
            subj_rr_df.at[index, columns['Rpeak_Tplus1']] = rpeak_Tplus1       # R-peak at T+1 after R-R interval of task event onset
            subj_rr_df.at[index, columns['Rpeak_Tplus2']] = rpeak_Tplus2       # R-peak at T+2 after R-R interval of task event onset
            subj_rr_df.at[index, columns['Rpeak_Tplus3']] = rpeak_Tplus3       # R-peak at T+3 after R-R interval of task event onset

            # Calculate R-R intervals before (T-2, T-1) and after (T+1, T+2, T+3)
            subj_rr_df.at[index, columns['RR_s_Tminus2']] = round((rpeak_Tminus1 - rpeak_Tminus2),3)
            subj_rr_df.at[index, columns['RR_s_Tminus1']] = round((rpeak_pre - rpeak_Tminus1),3)
            subj_rr_df.at[index, columns['RR_s_Tplus1']] = round((rpeak_Tplus1 - rpeak_post),3)
            subj_rr_df.at[index, columns['RR_s_Tplus2']] = round((rpeak_Tplus2 - rpeak_Tplus1),3)
            subj_rr_df.at[index, columns['RR_s_Tplus3']] = round((rpeak_Tplus3 - rpeak_Tplus2),3)

        beh_ecg_rr.append(subj_rr_df)

    # Concatenate the dfs for all participants
    beh_ecg_rr_prepost = pd.concat(beh_ecg_rr, ignore_index=True)
    beh_ecg_rr_prepost.rename(columns={'Unnamed: 0': 'subj_index'}, inplace=True)

    # Create df in wide format with mean R-R intervals pre- and post-task event
    rr_prepost_wide = beh_ecg_rr_prepost.groupby(['subjID'])[[f'RR_s_Tminus2_{abbrev}', f'RR_s_Tminus1_{abbrev}', 
                                                              f'RR_s_{abbrev}', f'RR_s_Tplus1_{abbrev}', 
                                                              f'RR_s_Tplus2_{abbrev}', f'RR_s_Tplus3_{abbrev}']].mean()
    
    # Create df in long format with R-R intervals duration for each trial and timepoint
    rr_prepost_long = beh_ecg_rr_prepost.melt(id_vars=[col for col in column_map.values()] + ['act_sys', 'act_dia'], 
                                              value_vars=['RR_s_Tminus2_act', 'RR_s_Tminus1_act', 'RR_s_act', 
                                                          'RR_s_Tplus1_act', 'RR_s_Tplus2_act', 'RR_s_Tplus3_act'],
                                              var_name='Timepoint', value_name='RR_Interval')
    
    # Replace timepoint names with shorter labels
    timepoint_labels = {
        f'RR_s_Tminus2_{abbrev}': 'T-2', f'RR_s_Tminus1_{abbrev}': 'T-1', f'RR_s_{abbrev}': 'T0',
        f'RR_s_Tplus1_{abbrev}': 'T+1', f'RR_s_Tplus2_{abbrev}': 'T+2', f'RR_s_Tplus3_{abbrev}': 'T+3'}
        
    rr_prepost_long['Timepoint'] = rr_prepost_long['Timepoint'].replace(timepoint_labels)
    
    rr_prepost_long = rr_prepost_long.dropna()  # drop NaNs of BasT condition and NoResp trials


    return beh_ecg_rr_prepost, rr_prepost_wide, rr_prepost_long

In [67]:
# Subselect only relevant columns fro beh-ecg df 
beh_ecg_action_rr = beh_ecg_action_clean[['subjID', 'condition', 'n_block', 'n_trial', 'real_report_diff_act_deg', 'JE_act',
                                          'keypress_onset', 'Rpeak_pre_act', 'Rpeak_post_act', 'RR_s_act', 'act_sys', 'act_dia']].copy()

# Extract R-R intervals before (T-2, T-1) and after (T+1, T+2, T+3) the R-R interval of the action (T0)
beh_ecg_rr_prepost, rr_prepost_wide, rr_prepost_long = extract_prepost_rr(beh_ecg_df=beh_ecg_action_rr, abbrev=abbrev)

In [68]:
# Define function for plotting changes in R-R intervals before and after the task event (T0)
# separately per each task condition

def plot_prepost_rr(rr_prepost_long, cond_colors, errorbar_type='CI', n_boot=1000, 
                    condition_col=column_map['condition']): 

    # Convert RR_Interval from sec to msec
    rr_prepost_long['RR_Interval_ms'] = rr_prepost_long['RR_Interval'] * 1000  # Convert to milliseconds

    # Plot the group average with confidence intervals
    plt.figure(figsize=(6, 6), dpi=300)

    # Extract errorbar type based on user input
    errorbar_arg = ('ci',95) if errorbar_type.lower() == 'CI' else ('se',2)

    # Plot lines for RR interval changes by condition
    sns.lineplot(data=rr_prepost_long, x='Timepoint', y='RR_Interval_ms', hue=condition_col,
                sort= False, marker='o', linewidth=2, palette=cond_colors, 
                err_style='band', errorbar=errorbar_arg, n_boot=n_boot)

    plt.xlabel("Timepoint relative to keypress onset (T0)", fontsize='x-large')
    plt.ylabel("Mean R-R Interval (ms)", fontsize='x-large')
    plt.xticks(ticks=[0,1,2,3,4,5], labels=['T-2', 'T-1', 'T0', 'T+1', 'T+2', 'T+3'])
    plt.ylim(800,870)
    sns.despine()

    # Save plot 
    fig10 = os.path.join(plotting_dir, "Figure6a_rr_changes_prepostaction.svg")
    plt.savefig(fig10, format="svg")

    plt.show()

In [ ]:
### FIGURE 6a ###

# Define color palette for IB task conditions
cond_colpalette = {'BasA': '#a4cbb6', 'OpA': '#007a4e', 'OpT': '#0b3142'}
 
# Plot pre- and post-event R-R interval changes per condition
plot_prepost_rr(rr_prepost_long=rr_prepost_long, cond_colors=cond_colpalette, errorbar_type='ci')

In [ ]:
### Two-way repeated-measure ANOVA 6 (Timepoint) x 3 (Condition) on R-R interval duration ###

# Aggregate by participant, timepoint and condition 
rr_prepost_mean = (
    rr_prepost_long.groupby(['subjID', 'Timepoint', 'condition'], as_index=False)['RR_Interval'].mean())

# Run rmANOVA
rr_prepost_anova = pg.rm_anova(
    dv='RR_Interval',
    within=['Timepoint', 'condition'],
    subject='subjID',
    data=rr_prepost_mean,
    detailed=True)

print(rr_prepost_anova)

In [71]:
# Only look at consecutive timepoints for main effect
timepoints_of_interest = [('T-2','T-1'), ('T-1','T0'), ('T0','T+1'), ('T+1','T+2'), ('T+2','T+3')]

timepoint_contrasts = []
for A, B in timepoints_of_interest:
    subdf = rr_prepost_long[(rr_prepost_long['Timepoint'].isin([A, B]))]
    res = pg.pairwise_tests(
        dv='RR_Interval',
        within='Timepoint',
        subject='subjID',
        data=subdf,
        padjust='bonf',
        effsize='cohen'
    )
    res['contrast'] = f"{A} vs {B}"
    timepoint_contrasts.append(res)

timepoint_contrasts = pd.concat(timepoint_contrasts, ignore_index=True)
timepoint_contrasts

,Contrast,A,B,Paired,Parametric,T,dof,alternative,p_unc,BF10,cohen,contrast
0,Timepoint,T-1,T-2,True,True,7.680896,43.0,two-sided,1.348338e-09,8.524e+06,0.069865,T-2 vs T-1
1,Timepoint,T-1,T0,True,True,-10.209409,43.0,two-sided,4.584651e-13,1.759e+10,-0.101441,T-1 vs T0
2,Timepoint,T+1,T0,True,True,-2.337456,43.0,two-sided,2.413948e-02,1.87,-0.026365,T0 vs T+1
3,Timepoint,T+1,T+2,True,True,6.178688,43.0,two-sided,2.009402e-07,7.535e+04,0.060796,T+1 vs T+2
4,Timepoint,T+2,T+3,True,True,2.791614,43.0,two-sided,7.792859e-03,4.881,0.026130,T+2 vs T+3


In [72]:
# Pairwise comparisons for Timepoint
timepoint_posthoc = pg.pairwise_tests(dv="RR_Interval", within=["Timepoint", "condition"],
                                      subject="subjID", data=rr_prepost_long,
                                      parametric=True, padjust="bonf")

# Display results
print(timepoint_posthoc)

                 Contrast Timepoint     A    B Paired Parametric          T  \
0               Timepoint         -   T+1  T+2   True       True   6.196882   
1               Timepoint         -   T+1  T+3   True       True   5.939289   
2               Timepoint         -   T+1  T-1   True       True   5.785548   
3               Timepoint         -   T+1  T-2   True       True   9.676881   
4               Timepoint         -   T+1   T0   True       True  -2.337029   
5               Timepoint         -   T+2  T+3   True       True   2.803233   
6               Timepoint         -   T+2  T-1   True       True   0.926529   
7               Timepoint         -   T+2  T-2   True       True   5.326451   
8               Timepoint         -   T+2   T0   True       True  -4.620728   
9               Timepoint         -   T+3  T-1   True       True  -0.487061   
10              Timepoint         -   T+3  T-2   True       True   3.429919   
11              Timepoint         -   T+3   T0   Tru

### **9b. Pre-action cardiac deceleration**

Perform slope analysis of cardiac deceleration (z-scored) preceding action onset (i.e., from T-1 to T0) between intentional binding conditions. 

In [74]:
############## 9b. Pre-action cardiac deceleration ##############

# Compute deceleration (difference in RR intervals between T-1 and T0)
beh_ecg_rr_prepost[f'pre{abbrev}_HRdecel'] = beh_ecg_rr_prepost[f'RR_s_{abbrev}'] - beh_ecg_rr_prepost[f'RR_s_Tminus1_{abbrev}']

# Standardize deceleration per participant (Z-score)
beh_ecg_rr_prepost[f'pre{abbrev}_HRdecel_zscore'] = beh_ecg_rr_prepost.groupby('subjID')[f'pre{abbrev}_HRdecel'].transform(lambda x: (x - x.mean()) / x.std())

# Compute mean deceleration per Condition (BasA, OpA, OpT) and participant
dec_data = beh_ecg_rr_prepost.groupby(['subjID', 'condition'])[f'pre{abbrev}_HRdecel_zscore'].mean().unstack().reset_index()
dec_data = dec_data.drop(columns=['BasT'])

# Convert to long format for seaborn
dec_data_long = dec_data.melt(id_vars=['subjID'], var_name='Condition', value_name='Deceleration')

### rmANOVA of condition on deceleration ###

# Run rmANOVA with deceleration (z-scores) as DV, Condition as within-subjects factor
rmanova_decel = pg.rm_anova(
    data=dec_data_long,
    dv="Deceleration",           # Dependent variable
    within="Condition",          # Within-subject factors
    subject="subjID",            # Participant ID 
    detailed=True, 
    correction=True              # Check for sphericity assumption
)
print(rmanova_decel)

# Summary of rmANOVA for cardiac deceleration per condition
md("rmANOVA showed no significant effect of condition on pre-action cardiac deceleration (z-scores) (F({}, {})={}, p={}, ng2={}).".format(
    rmanova_decel['DF'][0], rmanova_decel['DF'][1], round(rmanova_decel['F'].iloc[0],3),
    round(rmanova_decel['p_unc'].iloc[0],3), round(rmanova_decel['ng2'].iloc[0],3)
))

      Source        SS  DF        MS         F     p_unc  p_GG_corr       ng2  \
0  Condition  0.017691   2  0.008846  0.221611  0.801683   0.800603  0.005126   
1      Error  3.432683  86  0.039915       NaN       NaN        NaN       NaN   

        eps sphericity  W_spher  p_spher  
0  0.994995       True  0.99497  0.89951  
1       NaN        NaN      NaN      NaN  


rmANOVA showed no significant effect of condition on pre-action cardiac deceleration (z-scores) (F(2, 86)=0.222, p=0.802, ng2=0.005).

In [75]:
### rmANOVA of condition on deceleration - post-hoc comparisons ###

posthoc_decel = pg.pairwise_tests(dv="Deceleration", within="Condition", subject="subjID", data=dec_data_long, padjust="bonferroni")
print(posthoc_decel)

    Contrast     A    B  Paired  Parametric         T   dof alternative  \
0  Condition  BasA  OpA    True        True -0.165679  43.0   two-sided   
1  Condition  BasA  OpT    True        True -0.644729  43.0   two-sided   
2  Condition   OpA  OpT    True        True -0.463406  43.0   two-sided   

      p_unc  p_corr    p_adjust   BF10    hedges  
0  0.869186     1.0  bonferroni  0.165 -0.042149  
1  0.522528     1.0  bonferroni  0.199 -0.166650  
2  0.645411     1.0  bonferroni  0.181 -0.121977  


In [76]:
print("----- Mean (SD) pre-action deceleration -----")
print(dec_data_long.groupby('Condition')['Deceleration'].agg(['mean', 'std']))

----- Mean (SD) pre-action deceleration -----
               mean       std
Condition                    
BasA      -0.011440  0.157856
OpA       -0.004572  0.165110
OpT        0.015821  0.166338


### **9c. Post-action cardiac acceleration**

Perform slope analysis of cardiac acceleration (z-scored) following action onset (i.e., from T0 to T+1) between intentional binding conditions. 

In [78]:
############## 9c. Post-action cardiac acceleration ##############

# Compute acceleration (difference in RR intervals between T0 and T+1)
beh_ecg_rr_prepost[f'post{abbrev}_HRaccel'] = beh_ecg_rr_prepost[f'RR_s_Tplus1_{abbrev}'] - beh_ecg_rr_prepost[f'RR_s_{abbrev}']

# Standardize acceleration per participant (Z-score)
beh_ecg_rr_prepost[f'post{abbrev}_HRaccel_zscore'] = beh_ecg_rr_prepost.groupby('subjID')[f'post{abbrev}_HRaccel'].transform(lambda x: (x - x.mean()) / x.std())

# Compute mean acceleration per Condition (BasA, OpA, OpT) and participant
accel_data = beh_ecg_rr_prepost.groupby(['subjID', 'condition'])[f'post{abbrev}_HRaccel_zscore'].mean().unstack().reset_index()
accel_data = accel_data.drop(columns=['BasT'])

# Convert to long format for seaborn
accel_data_long = accel_data.melt(id_vars=['subjID'], var_name='Condition', value_name='Acceleration')

### rmANOVA of condition on acceleration ###

# Run rmANOVA with acceleration (z-scores) as DV, Condition as within-subjects factor
rmanova_accel = pg.rm_anova(
    data=accel_data_long,
    dv="Acceleration",          # Dependent variable
    within="Condition",         # Within-subject factors
    subject="subjID",           # Participant ID
    detailed=True,
    correction=True             # Check for sphericity assumption
)
print(rmanova_accel)

# Summary of rmANOVA for cardiac acceleration per condition
md("rmANOVA showed a significant effect of condition on post-action cardiac acceleration (z-scores) (F({}, {})={}, p={}, ng2={}).".format(
    rmanova_accel['DF'][0], rmanova_accel['DF'][1], round(rmanova_accel['F'].iloc[0],3),
    round(rmanova_accel['p_unc'].iloc[0],3), round(rmanova_accel['ng2'].iloc[0],3)
))

      Source        SS  DF        MS         F     p_unc  p_GG_corr       ng2  \
0  Condition  0.434153   2  0.217077  6.122995  0.003265   0.004324  0.124599   
1      Error  3.048932  86  0.035453       NaN       NaN        NaN       NaN   

        eps sphericity   W_spher   p_spher  
0  0.915469       True  0.907664  0.130745  
1       NaN        NaN       NaN       NaN  


rmANOVA showed a significant effect of condition on post-action cardiac acceleration (z-scores) (F(2, 86)=6.123, p=0.003, ng2=0.125).

In [79]:
### rmANOVA of condition on acceleration - post-hoc comparisons ###

posthoc_accel = pg.pairwise_tests(dv="Acceleration", within="Condition", subject="subjID", data=accel_data_long, padjust="bonferroni")
print(posthoc_accel)

    Contrast     A    B  Paired  Parametric         T   dof alternative  \
0  Condition  BasA  OpA    True        True -0.409994  43.0   two-sided   
1  Condition  BasA  OpT    True        True -2.955173  43.0   two-sided   
2  Condition   OpA  OpT    True        True -2.664085  43.0   two-sided   

      p_unc    p_corr    p_adjust   BF10    hedges  
0  0.683846  1.000000  bonferroni  0.177 -0.095592  
1  0.005054  0.015161  bonferroni  7.102 -0.795312  
2  0.010824  0.032471  bonferroni  3.681 -0.712840  


In [80]:
print("----- Mean (SD) post-action acceleration -----")
print(accel_data_long.groupby('Condition')['Acceleration'].agg(['mean', 'std']))

----- Mean (SD) post-action acceleration -----
               mean       std
Condition                    
BasA      -0.047087  0.143151
OpA       -0.033352  0.141698
OpT        0.080856  0.174256


In [81]:
# Save dataframes for pre- and post-action RR interval changes

decel_dir = os.path.join(cardphase_dir, 'task-CardiAgIBTask_preact_cardiac_deceleration.tsv')
dec_data_long.to_csv(decel_dir, index=False, sep='\t', na_rep="n/a")

accel_dir = os.path.join(cardphase_dir, 'task-CardiAgIBTask_postact_cardiac_acceleration.tsv')
accel_data_long.to_csv(accel_dir, index=False, sep='\t', na_rep="n/a")